In [ ]:
from dotenv import load_dotenv
import os, sys, json, re, warnings
from datetime import datetime
load_dotenv("")

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langgraph.graph import StateGraph, END
from langgraph.types import interrupt, Command
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict, Optional

SCRIPT_DIR  = os.path.dirname(os.path.abspath("__file__"))
sys.path.insert(0, SCRIPT_DIR)
from aa_constructor import (
    build_and_write, load_opls, calculate_box_size,
    validate_monomer_topology,
    DEFAULT_N_CHAINS, DEFAULT_N_MONOMERS, DEFAULT_DENSITY,
    DEFAULT_SEED, MAX_REVISIONS,
    compare_cru_formula, cru_formula,
)
from mapper import build_cg_data
from potential import compute_cg_potentials, compute_cg_potentials_multi
from simulator import (
    build_in_file, run_lammps, build_tg_in_file, build_aa_tg_in_file,
    calc_tg_temps,
    DEFAULT_T_INIT, DEFAULT_T_MAX, DEFAULT_N_ANNEAL, DEFAULT_EQUIL_PS,
)
from validator import (
    get_reference_tg, calc_tg_from_annealing,
    compare_aa_cg_density, compare_aa_cg_tg,
    DEFAULT_DENSITY_TOL_PCT, DEFAULT_TG_TOL_K,
)
from analyzer import parse_npt_log, calc_equilibrium_density
from copolymer import (
    CopolymerSpec, build_and_write_copoly,
    generate_sequence, graft_branch_positions,
)

OPLS_DIR    = os.path.join(SCRIPT_DIR, "opls-aa")
TEMPLATE_IN = os.path.join(SCRIPT_DIR, "in.AAEQ")
from cg_constructor import (
    build_cg_initial_data, build_cg_in_file,
    DEFAULT_T_INIT_CG, DEFAULT_T_MAX_CG, DEFAULT_N_ANNEAL_CG, DEFAULT_EQUIL_PS_CG,
)
CG_TEMPLATE_IN = os.path.join(SCRIPT_DIR, "in.CGEQ")
TG_TEMPLATE_IN    = os.path.join(SCRIPT_DIR, "in.CGTg")
AA_TG_TEMPLATE_IN = os.path.join(SCRIPT_DIR, "in.AATg")
OUTPUT_BASE = os.path.join(SCRIPT_DIR, "output")
os.makedirs(OUTPUT_BASE, exist_ok=True)

def make_session_dir() -> str:
    date_str = datetime.now().strftime("%y%m%d")
    idx = 1
    while True:
        path = os.path.join(OUTPUT_BASE, f"{date_str}_{idx:03d}")
        if not os.path.exists(path):
            os.makedirs(path)
            return path
        idx += 1

import numpy as _np


def _to_native(obj):
    """Recursively convert numpy scalars/arrays to built-in Python types."""
    if isinstance(obj, _np.generic): 
        return obj.item()
    if isinstance(obj, _np.ndarray):
        return [_to_native(x) for x in obj.tolist()]
    if isinstance(obj, dict):
        return {_to_native(k): _to_native(v) for k, v in obj.items()}
    if isinstance(obj, tuple):
        return tuple(_to_native(x) for x in obj)
    if isinstance(obj, list):
        return [_to_native(x) for x in obj]
    if isinstance(obj, set):
        return {_to_native(x) for x in obj}
    return obj


def safe_node(fn):
    """Wrap a graph node so its returned state is always msgpack-safe."""
    def _wrapped(state):
        return _to_native(fn(state))
    _wrapped.__name__ = getattr(fn, "__name__", "node")
    _wrapped.__doc__  = getattr(fn, "__doc__", None)
    return _wrapped

class Tee:
    def __init__(self, *files):
        self.files = files

    def write(self, obj):
        for f in self.files:
            f.write(obj)
            f.flush()

    def flush(self):
        for f in self.files:
            f.flush()

    def isatty(self) -> bool:
        return False

    def fileno(self) -> int:
        """Return a valid file descriptor; needed when subprocess inherits stdout."""
        for f in self.files:
            if hasattr(f, "fileno"):
                try:
                    return f.fileno()
                except Exception:
                    pass
        raise OSError("Tee has no underlying file descriptor")


from contextlib import contextmanager

@contextmanager
def tee_stdout(session_dir: str, filename: str = "cgmas_log.txt"):
    """Mirror stdout into <session_dir>/<filename> for the duration of a run.

    stdout is always restored, even if the run raises, so a failed session
    cannot leave the notebook writing into a closed file.
    """
    path = os.path.join(session_dir, filename)
    log_file = open(path, "w", encoding="utf-8")
    original = sys.stdout
    sys.stdout = Tee(original, log_file)
    try:
        yield path
    finally:
        sys.stdout = original
        log_file.close()

def _log(session_dir: str, role: str, content: str) -> None:
    path = os.path.join(session_dir, "chat_log.json")
    log  = []
    if os.path.exists(path):
        with open(path, encoding="utf-8") as f:
            try:   log = json.load(f)
            except Exception: log = []
    log.append({"role": role, "content": content})
    with open(path, "w", encoding="utf-8") as f:
        json.dump(log, f, indent=2, ensure_ascii=False)

def _log_once(session_dir: str, role: str, content: str) -> bool:
    """Log if this exact message isn't already in the log.
    Returns True (and logs) on first occurrence; False (no-op) on duplicates.
    Prevents double-print caused by LangGraph re-executing nodes after interrupt."""
    path = os.path.join(session_dir, "chat_log.json")
    log  = []
    if os.path.exists(path):
        with open(path, encoding="utf-8") as f:
            try:   log = json.load(f)
            except Exception: log = []
    _last_user_idx = max((i for i, e in enumerate(log)
                          if e.get("role") == "user"), default=-1)
    _current_turn = log[_last_user_idx + 1:]
    if any(e.get("role") == role and e.get("content") == content
           for e in _current_turn):
        return False
    log.append({"role": role, "content": content})
    with open(path, "w", encoding="utf-8") as f:
        json.dump(log, f, indent=2, ensure_ascii=False)
    return True

In [ ]:
# State definition

class LAMMPSState(TypedDict):
    session_dir:              str
    initial_input:            str
    polymer_name:             Optional[str]
    sim_params:               Optional[dict]
    poly_def:                 Optional[dict]
    datafile_path:            Optional[str]
    review_feedback:          Optional[str]
    final_datafile_path:      Optional[str]
    sim_script_params:        Optional[dict]
    in_file_path:             Optional[str]
    aa_fin_data_path:         Optional[str]
    bead_map_def:             Optional[dict]
    bead_def:                 Optional[dict]
    cg_data_path:             Optional[str]
    potential_results:        Optional[dict]
    cg_sim_params:            Optional[dict]
    cg_initial_data_path:     Optional[str]
    cg_in_file_path:          Optional[str]
    cg_sim_script_params:     Optional[dict]
    cg_topology:              Optional[dict]
    user_locked_topology:    Optional[dict]
    revision_count:           int
    potential_revision_count: int
    cg_data_build_meta:       Optional[dict]
    status:                   str
    aa_log_path:              Optional[str]
    cg_log_path:              Optional[str]
    tg_temps:                 Optional[dict] 
    aa_tg_in_path:            Optional[str]
    aa_tg_log_path:           Optional[str]
    tg_in_path:               Optional[str]  
    tg_log_path:              Optional[str]  
    aa_sim_tg:                Optional[float] 
    cg_sim_tg:                Optional[float] 
    sim_tg:                   Optional[float] 
    validation_summary:       Optional[dict]  
    calibration:              Optional[dict]
    poly_defs:               Optional[list]  
    copolymer_spec:          Optional[dict] 
    is_copolymer:            bool
    _copoly_components:      Optional[list]  
    _copoly_current_idx:     int          


In [ ]:
llm = ChatOpenAI(model="gpt-5.4-mini", temperature=0)

with open(os.path.join(OPLS_DIR, "atom.txt")) as f:
    OPLS_TABLE = f.read()

In [ ]:
try:
    from langchain_core.callbacks.base import BaseCallbackHandler
except Exception:
    from langchain.callbacks.base import BaseCallbackHandler

TOKEN_PRICES_USD_PER_1M = {
    "gpt-4o-mini":  (0.15, 0.60),
    "gpt-4o":       (2.50, 10.00),
    "gpt-4.1-mini": (0.40, 1.60),
    "gpt-4.1":      (2.00, 8.00),
    "gpt-5.4-nano": (0.15, 0.60),
}
_PLACEHOLDER_MODELS = {"gpt-5.4-mini"}

class TokenCostTracker(BaseCallbackHandler):
    def __init__(self):
        self.input_tokens = 0
        self.output_tokens = 0
        self.calls = 0

    def on_llm_end(self, response, **kwargs):
        self.calls += 1
        usage = None
        llm_out = getattr(response, "llm_output", None)
        if isinstance(llm_out, dict):
            usage = llm_out.get("token_usage") or llm_out.get("usage")
        if usage:
            self.input_tokens  += usage.get("prompt_tokens", usage.get("input_tokens", 0)) or 0
            self.output_tokens += usage.get("completion_tokens", usage.get("output_tokens", 0)) or 0
            return
        try:
            for gen_list in getattr(response, "generations", []):
                for gen in gen_list:
                    msg = getattr(gen, "message", None)
                    um  = getattr(msg, "usage_metadata", None) if msg is not None else None
                    if um:
                        self.input_tokens  += um.get("input_tokens", 0) or 0
                        self.output_tokens += um.get("output_tokens", 0) or 0
        except Exception:
            pass

    @property
    def total_tokens(self):
        return self.input_tokens + self.output_tokens
    
def _lookup_price(model_name):
    if model_name in TOKEN_PRICES_USD_PER_1M:
        return TOKEN_PRICES_USD_PER_1M[model_name], (model_name in _PLACEHOLDER_MODELS)
    for key, price in TOKEN_PRICES_USD_PER_1M.items():
        if model_name.startswith(key):
            return price, (key in _PLACEHOLDER_MODELS)
    return (0.15, 0.60), True


def emit_token_report(tracker, session_dir, model_name):
    """Compute cost, write token_usage.json + token_cost.txt into the session
    dir, log to chat_log, and return a printable text block."""
    (p_in, p_out), is_estimate = _lookup_price(model_name)
    in_cost  = tracker.input_tokens  / 1_000_000 * p_in
    out_cost = tracker.output_tokens / 1_000_000 * p_out
    total_usd = in_cost + out_cost

    note = "  (price is a PLACEHOLDER/ESTIMATE -- edit TOKEN_PRICES_USD_PER_1M)" if is_estimate else ""
    text = (
        "Token usage & cost\n"
        f"  Model           : {model_name}{note}\n"
        f"  LLM calls       : {tracker.calls}\n"
        f"  Input tokens    : {tracker.input_tokens:,}  @ ${p_in:.3f}/1M = ${in_cost:.8f}\n"
        f"  Output tokens   : {tracker.output_tokens:,}  @ ${p_out:.3f}/1M = ${out_cost:.8f}\n"
        f"  Total tokens    : {tracker.total_tokens:,}\n"
        f"  Total cost      : ${total_usd:.8f}  (~{total_usd*USD_TO_KRW:.4f} KRW @ {USD_TO_KRW:.0f}/$)"
    )

    data = {
        "model": model_name,
        "price_is_estimate": is_estimate,
        "price_usd_per_1M": {"input": p_in, "output": p_out},
        "llm_calls": tracker.calls,
        "input_tokens": tracker.input_tokens,
        "output_tokens": tracker.output_tokens,
        "total_tokens": tracker.total_tokens,
        "cost_usd": {"input": round(in_cost, 8), "output": round(out_cost, 8),
                     "total": round(total_usd, 8)},
        "cost_krw_total": round(total_usd * USD_TO_KRW, 4),
        "usd_to_krw": USD_TO_KRW,
    }
    try:
        with open(os.path.join(session_dir, "token_usage.json"), "w") as f:
            json.dump(data, f, indent=2)
        with open(os.path.join(session_dir, "token_cost.txt"), "w") as f:
            f.write(text + "\n")
    except Exception as e:
        print(f"[token] could not write report files: {e}")
    try:
        _log(session_dir, "cgmas", text)
    except Exception:
        pass
    return text

def _to_kelvin(value: float, unit: str) -> int:
    u = (unit or "k").strip().lower().lstrip("\u00b0")
    if u.startswith("c"):
        return int(round(value + 273.15))
    if u.startswith("f"):
        return int(round((value - 32.0) * 5.0 / 9.0 + 273.15))
    return int(round(value))

_T_PEAK_WORDS = ("max", "peak", "up temp", "anneal", "heat", "highest", "upper")
_T_UNIT = r"(?:\u00b0\s*)?([KkCcFf])\b"

def _regex_temperature_backstop(query: str, out: dict) -> dict:
    """Fill temperature directives the LLM missed, straight from the query text.

    Only writes keys that are still None, so an explicit LLM answer always wins.
    """
    q = query or ""
    m = re.search(r"(?:between\s+)?(\d{2,4})\s*(?:[KkCcFf]\b)?\s*(?:to|-|~|and)\s*"
                  r"(\d{2,4})\s*" + _T_UNIT, q)
    if m:
        lo, hi, unit = float(m.group(1)), float(m.group(2)), m.group(3)
        lo_k, hi_k = _to_kelvin(lo, unit), _to_kelvin(hi, unit)
        if hi_k < lo_k:
            lo_k, hi_k = hi_k, lo_k
        for k, v in (("aa_t_init", lo_k), ("cg_t_init", lo_k),
                     ("pot_temperature", lo_k), ("aa_t_max", hi_k), ("cg_t_max", hi_k)):
            if out.get(k) is None:
                out[k] = v
        return out

    for m in re.finditer(r"(\d{2,4}(?:\.\d+)?)\s*" + _T_UNIT, q):
        val, unit = float(m.group(1)), m.group(2)
        temp_k = _to_kelvin(val, unit)
        if not (1 <= temp_k <= 5000):
            continue
        ctx = q[max(0, m.start() - 40):m.start()].lower()
        if "tg" in ctx or "glass" in ctx:
            continue
        side_aa = "aa" in ctx or "all-atom" in ctx or "all atom" in ctx
        side_cg = "cg" in ctx or "coarse" in ctx
        is_peak = any(w in ctx for w in _T_PEAK_WORDS)
        tgt = "t_max" if is_peak else "t_init"
        sides = ("aa",) if (side_aa and not side_cg) else \
                ("cg",) if (side_cg and not side_aa) else ("aa", "cg")
        for sd_ in sides:
            if out.get(f"{sd_}_{tgt}") is None:
                out[f"{sd_}_{tgt}"] = temp_k
        if not is_peak and "aa" in sides and out.get("pot_temperature") is None:
            out["pot_temperature"] = temp_k
    return out


_TEMP_DIRECTIVE_MAP = {
    "sim_params":    {"t_init": "aa_t_init", "t_max": "aa_t_max"},
    "cg_sim_params": {"t_init": "cg_t_init", "t_max": "cg_t_max"},
}

def _apply_temp_directives(state, key, params):
    """Force the query's temperatures into a param dict (no-op if none given)."""
    d = state.get("directives") or {}
    mapping = _TEMP_DIRECTIVE_MAP.get(key, {})
    out = dict(params)
    for field, dkey in mapping.items():
        val = d.get(dkey)
        if val is not None:
            out[field] = int(val)
    if out.get("t_max") is not None and out.get("t_init") is not None \
            and out["t_max"] < out["t_init"]:
        out["t_max"] = out["t_init"] + 200
    return out

def _auto_param_reply(state, key):
    """Build a precise per-node instruction from the parsed directives."""
    d = state.get("directives") or {}
    parts = []
    if key in ("aa_params", "cg_params"):
        dens = d.get("aa_density") if key == "aa_params" else d.get("cg_density")
        if dens is not None:
            parts.append(f"set the density to {dens}")
        if d.get("n_monomers") is not None:
            parts.append(f"set the chain length (monomers per chain) to {d['n_monomers']}")
        if d.get("n_chains") is not None:
            parts.append(f"set the number of chains to {d['n_chains']}")
    elif key in ("sim_params", "cg_sim_params"):
        aa = (key == "sim_params")
        eq = d.get("aa_equil_ps") if aa else d.get("cg_equil_ps")
        if eq is not None:
            parts.append(f"set the total equilibrium time to {eq} ps")
        t_i = d.get("aa_t_init") if aa else d.get("cg_t_init")
        t_m = d.get("aa_t_max")  if aa else d.get("cg_t_max")
        if t_i is not None:
            parts.append(f"set the operating temperature t_init to {t_i} K")
        if t_m is not None:
            parts.append(f"set the annealing peak temperature t_max to {t_m} K")
    if parts:
        return "Please change the following: " + "; ".join(parts) + \
               ". Keep every other value at its current default."
    return "approve, use the current default values"

In [ ]:
# Greet node  --  polymer selection

def greet_node(state: LAMMPSState) -> LAMMPSState:
    sd  = state["session_dir"]
    msg = "Hello! Which polymer would you like to study?"
    if _log_once(sd, "cgmas", msg):
        print("\n" + "="*60)
        print("CGMas: " + msg)
        print("="*60)

    polymer_input = _resume(state, "polymer_name")
    _log(sd, "user", polymer_input)

    resp = llm.invoke([
        SystemMessage(content=(
            "Extract the polymer name from the user message. "
            "Return ONLY the clean polymer name in English "
            "(e.g. 'polystyrene', 'poly(ethylene oxide)', "
            "'poly(methyl methacrylate)'). No extra text."
        )),
        HumanMessage(content=polymer_input),
    ])
    polymer_name = resp.content.strip()

    reply = "Got it -- " + polymer_name + ". Building topology..."
    print("\nCGMas: " + reply)
    _log(sd, "cgmas", reply)

    return {**state, "polymer_name": polymer_name, "status": "polymer_confirmed"}

In [ ]:
# Copolymer spec node  --  detect architecture after greet

COPOLY_PARSE_SYSTEM = """You are helping set up a polymer simulation.

Given a polymer name/description, determine:
1. Is this a copolymer (two or more chemically distinct repeat units)?
2. If yes: list the component polymers, architecture, and composition.

Respond with EXACTLY one JSON object (no markdown):
{
  "is_copolymer": <bool>,
  "components": ["<polymer_A>", "<polymer_B>", ...],
  "architecture": "<homopolymer|random|alternating|block|periodic|graft>",
  "fractions": [<float>, ...],
  "block_lengths": [],
  "pattern": [],
  "graft_interval": 4,
  "graft_length": 3
}

For homopolymer set is_copolymer=false, components=[full name], architecture="homopolymer".
For random EP 50/50: fractions=[0.5,0.5].
For AABAABAAB pattern: pattern=[0,0,1]."""


def copolymer_spec_node(state: LAMMPSState) -> LAMMPSState:
    sd = state["session_dir"]

    resp = llm.invoke([
        SystemMessage(content=COPOLY_PARSE_SYSTEM),
        HumanMessage(content=state["polymer_name"]),
    ])
    raw   = resp.content.strip()
    clean = re.sub(r'^```(?:json)?\s*|\s*```$', '', raw).strip()
    try:
        parsed = json.loads(clean)
    except Exception:
        parsed = {"is_copolymer": False, "components": [state["polymer_name"]],
                  "architecture": "homopolymer", "fractions": [1.0],
                  "block_lengths": [], "pattern": [], "graft_interval": 4, "graft_length": 3}

    is_copoly  = parsed.get("is_copolymer", False)
    components = parsed.get("components", [state["polymer_name"]])

    if is_copoly:
        arch  = parsed.get("architecture", "random")
        fracs = parsed.get("fractions", [round(1/len(components), 3)] * len(components))
        msg   = (f"I'll model this as a {arch} copolymer of: {', '.join(components)}. "
                 f"Fractions: {[round(f,2) for f in fracs]}. Correct?")
        if _log_once(sd, "cgmas", msg):      # guard: skip on LangGraph resume re-run
            print("\n" + "="*60)
            print("CGMas: " + msg)
            print("="*60)

        user_reply = _resume(state, "copoly_confirm", auto_reply="yes, that is correct")
        _log(sd, "user", user_reply)

        fix_resp = llm.invoke([
            SystemMessage(content=(
                "User may correct the copolymer spec. Current spec: " + clean + ". "
                "If user approves -> return the same JSON unchanged. "
                "If user corrects -> return updated JSON. No markdown."
            )),
            HumanMessage(content=user_reply),
        ])
        fix_raw = re.sub(r'^```(?:json)?\s*|\s*```$', '', fix_resp.content.strip()).strip()
        try:
            parsed = json.loads(fix_raw)
        except Exception:
            pass

    _raw_fracs = parsed.get("fractions") or []
    _n_comp    = len(parsed.get("components", [state["polymer_name"]]))
    if not _raw_fracs or len(_raw_fracs) != _n_comp or sum(_raw_fracs) < 1e-6:
        _raw_fracs = [round(1.0 / _n_comp, 6)] * _n_comp

    spec_dict = {
        "architecture":  parsed.get("architecture", "homopolymer"),
        "fractions":     _raw_fracs,
        "block_lengths": parsed.get("block_lengths", []),
        "pattern":       parsed.get("pattern",       []),
        "graft_interval":parsed.get("graft_interval", 4),
        "graft_length":  parsed.get("graft_length",   3),
    }

    return {
        **state,
        "is_copolymer":          parsed.get("is_copolymer", False),
        "poly_defs":             None,
        "copolymer_spec":        spec_dict,
        "_copoly_components":    parsed.get("components", [state["polymer_name"]]),
        "_copoly_current_idx":   0,
        "status":                "copoly_spec_done",
    }

In [ ]:
# Formatter agent  --  poly_def JSON generation

DEFINER_SYSTEM = """You are an expert in polymer chemistry and OPLS-AA force field parametrization.

Given a polymer name, produce a JSON topology for one constitutional repeat unit (CRU)
suitable for all-atom LAMMPS simulation.

==== OPLS-AA ATOM TABLE ====
{opls_table}
============================

Output EXACTLY one valid JSON object (no markdown fences, no extra text):
{{
  "polymer_name": "<full IUPAC name>",
  "description":  "<1-sentence description>",
  "monomer_atoms": [
    {{
      "local_id": <int, 1-based>,
      "opls_id":  "<unique_key from table>",
      "charge":   <float, from table>,
      "comment":  "<chemical environment>"
    }}
  ],
  "intra_bonds": [[i, j], ...],
  "head_atom":   <int>,
  "tail_atom":   <int>,
  "notes": "<any force-field caveats>"
}}

────────────────────────────────────────────────────────────────
Reasoning guide

What is a CRU?
  The constitutional repeat unit is the smallest fragment whose
  repetition reproduces the polymer chain. Start from the IUPAC name
  or the polymerization mechanism to identify it — do not include
  atoms that belong to the next or previous repeat unit.

How to select an OPLS type?
  Each entry in the table carries a comment describing its chemical
  environment. Read the hybridization, neighboring atoms, and
  functional group context of each atom in your structure, then pick
  the table entry whose comment best describes that environment.
  Partial charges in OPLS-AA reflect the local electronic environment:
  atoms near electronegative neighbors tend to carry more positive
  charge, isolated sp3 C–H groups carry small charges. Use those
  trends to sanity-check your choices.

Charge neutrality?
  A neutral polymer's repeat unit should sum to ~0 e.
  If your sum is non-zero, reconsider the atoms closest to
  heteroatoms or π-systems — those are where misassignments
  are most common.

Ring structures?
  Include all ring atoms and the ring-closing bond in intra_bonds.
  head_atom and tail_atom are the attachment points to adjacent CRUs.

Double bonds & unsaturation?
  A 1,4-polymerised diene REPEAT UNIT (polybutadiene, polyisoprene,
  polychloroprene, ...) has exactly ONE backbone C=C -- NOT the two
  C=C the free monomer had. So of its four backbone carbons, EXACTLY
  TWO (the single internal C=C pair) are sp2 and use the alkene carbon
  type (bond_type "CM") with alkene H (bond_type "HC", comment
  "alkene H"); the OTHER TWO backbone carbons are saturated -CH2- and
  MUST stay sp3 CT with alkane H. Two mistakes to avoid:
    (a) typing EVERY backbone carbon as CM -- that is the free-monomer
        diene (two C=C), not the polymer repeat unit, and it will not
        reach charge neutrality;
    (b) dropping the C=C entirely and making the unit fully saturated
        -CH2-CH2-.
  The alkene types ARE in the table above; do not claim they are missing.
  Example -- 1,4-polybutadiene CRU -CH2-CH=CH-CH2- = 2 sp3 CH2 (CT) +
  2 sp2 =CH (CM) + 4 alkane H + 2 alkene H  (~0 e).
  Example -- 1,4-polychloroprene CRU -CH2-C(Cl)=CH-CH2- = 2 sp3 CH2 (CT)
  + 2 sp2 C (CM: one bears the Cl, one bears H) + Cl + the H atoms.

Branched & disubstituted repeat units?
  Count the substituents on EVERY backbone carbon before writing the
  atom list. A backbone carbon bearing TWO alkyl branches is
  QUATERNARY (zero H); one bearing ONE branch is a methine (one H).
  Dropping a branch silently converts the polymer into a different,
  perfectly valid polymer -- the unit still has correct valence and
  still sums to ~0 e, so no downstream check will catch it.
  Examples of 1,1-disubstituted repeat units (BOTH substituents
  required, backbone C is quaternary):
    polyisobutylene       -CH2-C(CH3)2-        C4H8   (TWO methyls)
    poly(methyl methacrylate) -CH2-C(CH3)(COOCH3)-   (methyl AND ester)
    poly(alpha-methylstyrene) -CH2-C(CH3)(C6H5)-     (methyl AND phenyl)
  Contrast the MONO-substituted ones (backbone C is a methine, one H):
    polypropylene         -CH2-CH(CH3)-        C3H6   (ONE methyl)
    polystyrene           -CH2-CH(C6H5)-
  Before returning, state the repeat unit's molecular formula to
  yourself and confirm it matches the polymer you were asked for.

If reviewer feedback is provided, address each issue while keeping
the user's original intent.
"""

_FORMULA_SYSTEM = """You are a polymer chemist. Given a polymer name, reply with the
molecular formula of ONE constitutional repeat unit (CRU) -- the smallest fragment
whose repetition builds the chain -- and nothing else.

Rules:
- Hill notation, carbon first, then hydrogen, then other elements alphabetically.
- The CRU, not the free monomer: addition polymers have the SAME formula as their
  monomer (ethylene -> C2H4), but condensation polymers LOSE the small molecule
  (e.g. a polyamide repeat unit excludes the water that was eliminated).
- Count substituents carefully: polyisobutylene -CH2-C(CH3)2- is C4H8, while
  polypropylene -CH2-CH(CH3)- is C3H6.
- Output ONLY the formula, e.g. C4H8. No prose, no explanation, no units.
- If you are not confident which polymer is meant, output UNKNOWN."""


def get_expected_cru_formula(polymer_name: str, llm) -> str:
    """Expected CRU molecular formula for a polymer name, or '' if unavailable."""
    try:
        resp = llm.invoke([SystemMessage(content=_FORMULA_SYSTEM),
                           HumanMessage(content=str(polymer_name))])
        text = (resp.content if hasattr(resp, "content") else str(resp)).strip()
        text = text.split()[0].strip().strip(".,;:`") if text else ""
        if not text or text.upper().startswith("UNKNOWN"):
            return ""
        return text if re.match(r"^[A-Z][a-z]?\d*", text) else ""
    except Exception as e:
        print(f"  [formula] expected-formula lookup failed ({e}); check skipped")
        return ""


def _format_opls_table(opls_atoms: dict) -> str:
    """Format atom params dict as a readable table for the LLM prompt."""
    lines = [f"{{'key':<12}} {{'bond_type':<10}} {{'mass':>8}} {{'charge':>8}} {{'sigma':>8}} {{'epsilon':>10}}  comment"]
    lines.append("-" * 75)
    for key, v in opls_atoms.items():
        lines.append(
            f"{key:<12} {v['bond_type']:<10} {v['mass']:>8.3f} {v['charge']:>8.4f}"
            f" {v['sigma']:>8.4f} {v['epsilon']:>10.6f}  {v.get('comment','')}"
        )
    return "\n".join(lines)



def _build_charge_feedback(poly_def, opls, net_q, expected):
    """Per-atom charge feedback with bond-neighbor context and mismatch hints."""
    H_EXPECTS_CM = {"HC_2"}   # alkene H: only valid on CM (sp2 C=C)
    H_EXPECTS_CA = {"HA_1"}   # aryl H: only valid on CA (aromatic)

    atoms  = poly_def.get('monomer_atoms', [])
    bonds  = poly_def.get('intra_bonds', [])
    id_map = {a['local_id']: a for a in atoms}
    adj    = {}
    for (i, j) in bonds:
        adj.setdefault(i, []).append(j)
        adj.setdefault(j, []).append(i)

    lines      = []
    mismatches = []
    for a in atoms:
        oid = a['opls_id']
        q   = opls['atom'][oid]['charge']
        cmt = opls['atom'][oid].get('comment', '')
        bt  = opls['atom'][oid]['bond_type']
        nbr_desc = []
        for nb_id in adj.get(a['local_id'], []):
            nb = id_map.get(nb_id)
            if not nb:
                continue
            nb_oid = nb['opls_id']
            nb_bt  = opls['atom'][nb_oid]['bond_type']
            nbr_desc.append(f"{nb_oid}({opls['atom'][nb_oid].get('comment','')[:24]})")
            # Flag H whose comment implies sp2/aromatic bonded to sp3 carbon
            if oid in H_EXPECTS_CM and not nb_bt.startswith('CM'):
                hc1_q = opls['atom'].get('HC_1', {}).get('charge', 0.06)
                delta = round(q - hc1_q, 4)
                exact = abs(abs(delta) - abs(net_q)) < 0.002
                alts  = [k for k, v in opls['atom'].items()
                          if v['bond_type'] == bt and k != oid][:3]
                alt_s = ', '.join(
                    f"{k}(q={opls['atom'][k]['charge']:+.4f},'{opls['atom'][k].get('comment','')[:20]}')"
                    for k in alts
                )
                msg = (f"  local_id={a['local_id']} {oid}('{cmt}')"
                       f" is bonded to {nb_oid}(type={nb_bt}), which is sp3, not alkene CM.")
                if exact:
                    msg += (f" Its charge excess vs HC_1: {delta:+.4f}"
                            f" equals the full net error {net_q:+.4f}.")
                if alt_s:
                    msg += f" Alternative {bt} types: {alt_s}."
                mismatches.append(msg)
            if oid in H_EXPECTS_CA and not nb_bt.startswith('CA'):
                mismatches.append(
                    f"  local_id={a['local_id']} {oid}('{cmt}')"
                    f" bonded to {nb_oid}(type={nb_bt}), expected aromatic CA."
                )
        lines.append(
            f"  local_id={a['local_id']} opls_id={oid} q={q:+.4f} ({cmt})"
            + (f"  bonded to: [{', '.join(nbr_desc)}]" if nbr_desc else "")
        )

    head_id = poly_def.get("head_atom")
    tail_id = poly_def.get("tail_atom")
    _H_BOND_TYPES = {"H", "HO", "HC", "HA", "HW"} 
    for a in atoms:
        lid  = a["local_id"]
        oid  = a["opls_id"]
        bt   = opls["atom"][oid]["bond_type"] if oid in opls.get("atom", opls) else ""
        if not bt.startswith("CT"): 
            continue
        neighbors    = adj.get(lid, [])
        h_nb_ids     = [nid for nid in neighbors
                        if any(opls["atom"].get(id_map[nid]["opls_id"], {})
                               .get("bond_type", "").startswith(p) for p in _H_BOND_TYPES)
                        if nid in id_map]
        non_h_count  = len(neighbors) - len(h_nb_ids)
        inter_bond   = 1 if lid in (head_id, tail_id) else 0
        expected_h   = 4 - non_h_count - inter_bond
        actual_h     = len(h_nb_ids)
        if actual_h > expected_h:
            role = ("HEAD" if lid == head_id else
                    "TAIL" if lid == tail_id else "")
            mismatches.append(
                f"  *** local_id={lid} {oid} is the {role} atom — it bonds to the "
                f"adjacent repeat unit, so its free valence for H is "
                f"4 − {non_h_count}(non-H bonds) − 1(inter-CRU bond) = {expected_h}. "
                f"It currently has {actual_h} H neighbor(s); "
                f"remove {actual_h - expected_h} excess HC atom(s) from the CRU."
            )

    _O_BOND_TYPES = {"O", "OH", "OS", "OY"}
    for (bi, bj) in poly_def.get("intra_bonds", []):
        ai = id_map.get(bi); aj = id_map.get(bj)
        if not ai or not aj: continue
        bt_i = opls["atom"].get(ai["opls_id"], {}).get("bond_type", "")
        bt_j = opls["atom"].get(aj["opls_id"], {}).get("bond_type", "")
        i_is_O = any(bt_i.startswith(p) for p in _O_BOND_TYPES)
        j_is_O = any(bt_j.startswith(p) for p in _O_BOND_TYPES)
        if i_is_O and j_is_O:
            mismatches.append(
                f"  *** IMPOSSIBLE bond: local_id={bi} {ai['opls_id']} (O-type) "
                f"— local_id={bj} {aj['opls_id']} (O-type). "
                f"Two oxygen atoms cannot bond directly in a normal organic polymer. "
                f"Remove the spurious O atom (likely {ai['opls_id'] if 'S' in ai['opls_id'] else aj['opls_id']}) "
                f"from the CRU entirely."
            )

    _CT_O_REQUIRED = {
        "CT_5":  1,  
        "CT_6":  2,
        "CT_16": 1,  
        "CT_17": 1,  
    }
    for a in atoms:
        required_o = _CT_O_REQUIRED.get(a["opls_id"])
        if required_o is None: continue
        o_neighbors = [nid for nid in adj.get(a["local_id"], [])
                       if nid in id_map and
                       any(opls["atom"].get(id_map[nid]["opls_id"], {})
                           .get("bond_type", "").startswith(p) for p in _O_BOND_TYPES)]
        actual_o = len(o_neighbors)
        if actual_o < required_o:
            cmt = opls["atom"].get(a["opls_id"], {}).get("comment", "")
            mismatches.append(
                f"  *** local_id={a['local_id']} {a['opls_id']} ('{cmt}') "
                f"requires {required_o} O neighbor(s) by its type definition, "
                f"but has {actual_o} in this CRU. "
                f"Replace with a plain alkane CT type: "
                f"CT_1(CH3,q=-0.18), CT_2(CH2,q=-0.12), CT_3(CH,q=-0.06), CT_4(C,q=0.00). "
                f"For PVA backbone CH2 use CT_2."
            )

    for a in atoms:
        lid = a["local_id"]
        oid = a["opls_id"]
        bt  = opls["atom"].get(oid, {}).get("bond_type", "")
        if not bt.startswith("CT"):
            continue
        if lid in (head_id, tail_id):
            continue  
        neighbors   = adj.get(lid, [])
        h_nb        = [nid for nid in neighbors if nid in id_map and
                       any(opls["atom"].get(id_map[nid]["opls_id"], {})
                           .get("bond_type", "").startswith(p) for p in _H_BOND_TYPES)]
        non_h       = [nid for nid in neighbors if nid not in set(h_nb)]
        all_c_nbrs  = all(
            opls["atom"].get(id_map[nid]["opls_id"], {})
                        .get("bond_type", "").startswith("CT")
            for nid in non_h if nid in id_map
        )
        if len(h_nb) == 0 and len(non_h) >= 2 and all_c_nbrs:
            cmt = opls["atom"].get(oid, {}).get("comment", "")
            mismatches.append(
                f"  *** SPURIOUS BACKBONE CARBON: local_id={lid} {oid} ('{cmt}') "
                f"has 0 H and bonds only to other carbons "
                f"({[id_map[n]['opls_id'] for n in non_h if n in id_map]}). "
                f"This indicates the LLM generated a multi-unit CRU (dimer/trimer) "
                f"instead of the single constitutional repeat unit (CRU). "
                f"For poly(vinyl alcohol), the CRU is exactly [-CH2-CH(OH)-] — "
                f"only 2 backbone carbons (CT_2 as head, CT_7 as tail). "
                f"Remove this spurious CT atom entirely from monomer_atoms and intra_bonds, "
                f"and set head_atom=CT_2 and tail_atom=CT_7 directly."
            )

    ct7_lids = [a["local_id"] for a in atoms
                if opls["atom"].get(a["opls_id"], {}).get("bond_type","") == "CT"
                and a["opls_id"] == "CT_7"]
    ct3_lids = [a["local_id"] for a in atoms
                if opls["atom"].get(a["opls_id"], {}).get("bond_type","") == "CT"
                and a["opls_id"] == "CT_3"]
    if ct7_lids and ct3_lids:
        for c3_lid in ct3_lids:
            c3_neighbors = [id_map[n]["opls_id"] for n in adj.get(c3_lid, []) if n in id_map]
            mismatches.append(
                f"  *** CT_3 + CT_7 co-occurrence: local_id={c3_lid} CT_3 "
                f"('{opls['atom'].get('CT_3',{}).get('comment','')}', q=-0.06) "
                f"should NOT appear alongside CT_7 in a simple alcohol polymer CRU. "
                f"CT_7 already covers the CHOH sp3 carbon. "
                f"Remove local_id={c3_lid} CT_3 entirely. "
                f"For PVA, the correct CRU is exactly: "
                f"CT_2(head) bonded to CT_7, with CT_7 bonded to OH_1-HO_1, "
                f"plus HC_1 × 2 on CT_2 and HC_1 × 1 on CT_7 (7 atoms total, "
                f"net charge = -0.12+0.205-0.683+0.418+0.06+0.06+0.06 = 0.00 e)."
            )

    for a in atoms:
        lid = a["local_id"]
        oid = a["opls_id"]
        bt  = opls["atom"].get(oid, {}).get("bond_type", "")
        if not bt.startswith("CT"):
            continue
        neighbors   = adj.get(lid, [])
        h_nb        = [nid for nid in neighbors if nid in id_map and
                       any(opls["atom"].get(id_map[nid]["opls_id"], {})
                           .get("bond_type", "").startswith(p) for p in _H_BOND_TYPES)]
        non_h_count = len(neighbors) - len(h_nb)
        inter_bond  = 1 if lid in (head_id, tail_id) else 0
        expected_h  = 4 - non_h_count - inter_bond
        actual_h    = len(h_nb)
        missing     = expected_h - actual_h
        if missing > 0:
            role = ("HEAD" if lid == head_id else
                    "TAIL" if lid == tail_id else "inner")
            mismatches.append(
                f"  *** MISSING H on {role} atom local_id={lid} {oid}: "
                f"has {actual_h} H neighbor(s) but needs {expected_h} "
                f"(4 valence − {non_h_count} non-H bonds − {inter_bond} inter-CRU bond). "
                f"Add {missing} HC_1 atom(s) (q=+0.0600 each) bonded to local_id={lid}. "
                f"Each missing HC_1 contributes +0.06 e to the net charge."
            )

    deficit = round(expected - net_q, 6)
    if abs(deficit) > 1e-4:
        swap_hints = []
        atom_table = opls.get("atom", opls)
        for a in atoms:
            oid     = a["opls_id"]
            cur_q   = atom_table[oid]["charge"]
            cur_bt  = atom_table[oid].get("bond_type", "")
            target_q = round(cur_q + deficit, 6)
            candidates = [
                (k, v) for k, v in atom_table.items()
                if k != oid
                and abs(v["charge"] - target_q) < 0.001
                and v.get("bond_type", "")[:2] == cur_bt[:2] 
            ]
            for ckey, cv in candidates:
                swap_hints.append(
                    f"  → Change local_id={a['local_id']} from {oid} "
                    f"(q={cur_q:+.4f}, '{atom_table[oid].get('comment','')}') "
                    f"to {ckey} (q={cv['charge']:+.4f}, '{cv.get('comment','')}') "
                    f"— this would cancel the {deficit:+.4f} e imbalance."
                )
        if swap_hints:
            mismatches.append(
                "  IMPORTANT — charges are always set from the table, so you MUST fix "
                "the atom TYPE (opls_id), not the charge value. Suggested type swap(s):"
            )
            mismatches.extend(swap_hints[:3])
        else:
            mismatches.append(
                f"  IMPORTANT — charges are always set from the table. "
                f"Net error = {deficit:+.4f} e. Reconsider which atom types you are "
                f"using: a wrong CT subtype (e.g. CT_3 vs CT_2) is a common cause."
            )

    out = (f"net charge = {net_q:+.4f} e (expected {expected:+.4f}).\n"
           f"Per-atom charges with bond neighbors:\n" + "\n".join(lines))
    if mismatches:
        out += "\n\nType-environment mismatches:\n" + "\n".join(mismatches)
    return out


_H_HYBRIDIZATION_RULES = {
    "HC_2": {"requires": "CM",  "fallback": "HC_1"},
    "HA_1": {"requires": "CA",  "fallback": None}, 
}

_H_PARENT_KEY_RULES = {
    "HC_3": {
        "valid_parents": {"CT_5"}, 
        "fallback": "HC_1",
        "reason": "ether H (PEO/PPO context) — only valid on CT_5; use HC_1 for plain alkane H",
    },
    "HC_4": {
        "valid_parents": {"CT_6"},     
        "fallback": "HC_1",
        "reason": "acetal H (POM context) — only valid on CT_6; use HC_1 for plain alkane H",
    },
    "HC_5": {
        "valid_parents": {"CT_14", "CT_15"},
        "fallback": "HC_1",
        "reason": "chloromethyl H — only valid on CH2Cl/CHCl carbons; use HC_1 otherwise",
    },
}


def _auto_fix_h_types(poly_def: dict, opls_atoms: dict) -> list:
    """Correct H types used on wrong-hybridization carbons (code-level enforcement)."""
    atoms  = poly_def.get("monomer_atoms", [])
    bonds  = poly_def.get("intra_bonds", [])
    id_map = {a["local_id"]: a for a in atoms}
    adj    = {}
    for (i, j) in bonds:
        adj.setdefault(i, []).append(j)
        adj.setdefault(j, []).append(i)
    corrections = []
    for a in atoms:
        rule = _H_HYBRIDIZATION_RULES.get(a["opls_id"])
        if not rule or not rule["fallback"] or rule["fallback"] not in opls_atoms:
            continue
        for nb_id in adj.get(a["local_id"], []):
            nb = id_map.get(nb_id)
            if not nb:
                continue
            nb_bt = opls_atoms[nb["opls_id"]]["bond_type"]
            if not nb_bt.startswith(rule["requires"]):
                old = a["opls_id"]
                a["opls_id"] = rule["fallback"]
                a["charge"]  = opls_atoms[rule["fallback"]]["charge"]
                corrections.append(f"local_id={a['local_id']} {old} → {rule['fallback']} (on {nb['opls_id']}, type={nb_bt})")
                break
    for a in atoms:
        key_rule = _H_PARENT_KEY_RULES.get(a["opls_id"])
        if not key_rule or key_rule["fallback"] not in opls_atoms:
            continue
        for nb_id in adj.get(a["local_id"], []):
            nb = id_map.get(nb_id)
            if not nb:
                continue
            if nb["opls_id"] not in key_rule["valid_parents"]:
                old = a["opls_id"]
                a["opls_id"] = key_rule["fallback"]
                a["charge"]  = opls_atoms[key_rule["fallback"]]["charge"]
                corrections.append(
                    f"local_id={a['local_id']} {old} → {key_rule['fallback']} "
                    f"(on {nb['opls_id']}; {key_rule['reason']})"
                )
                break
    return corrections


def _auto_fix_valence(poly_def: dict, opls_atoms: dict) -> list:
    """Remove excess H atoms from over-valenced carbons (esp. head/tail atoms).

    head/tail atoms carry one extra bond to the adjacent CRU, so their
    allowed H count = 4 - (non_H_intra_bonds) - 1.
    Regular sp3 carbons: allowed H = 4 - (non_H_intra_bonds).

    If more H atoms are present than allowed, the extras are deleted from
    monomer_atoms and intra_bonds.  This fixes the case where the LLM adds
    e.g. 3 H to a head-CH2 (should be 2) or 2 H to a tail-CHOH (should be 1).
    """
    atoms   = poly_def.get("monomer_atoms", [])
    bonds   = poly_def.get("intra_bonds", [])
    head_id = poly_def.get("head_atom")
    tail_id = poly_def.get("tail_atom")

    id_map = {a["local_id"]: a for a in atoms}
    adj    = {}
    for (i, j) in bonds:
        adj.setdefault(i, []).append(j)
        adj.setdefault(j, []).append(i)

    _H_BT = {"H", "HO", "HC", "HA", "HW", "HS"}

    def is_H(local_id):
        a = id_map.get(local_id)
        if not a: return False
        return any(opls_atoms.get(a["opls_id"], {})
                   .get("bond_type", "").startswith(p) for p in _H_BT)

    corrections = []
    ids_to_remove = set()

    for a in atoms:
        lid = a["local_id"]
        oid = a["opls_id"]
        bt  = opls_atoms.get(oid, {}).get("bond_type", "")
        if not bt.startswith("CT"):
            continue

        neighbors   = adj.get(lid, [])
        h_nb        = [nid for nid in neighbors if is_H(nid)]
        non_h_count = len(neighbors) - len(h_nb)
        inter_bond  = 1 if lid in (head_id, tail_id) else 0
        expected_h  = 4 - non_h_count - inter_bond
        excess      = len(h_nb) - expected_h

        if excess <= 0:
            continue

        role = ("HEAD" if lid == head_id else
                "TAIL" if lid == tail_id else "inner")
        to_del = h_nb[-excess:]
        for del_id in to_del:
            ids_to_remove.add(del_id)
            corrections.append(
                f"Removed excess H local_id={del_id} "
                f"({id_map[del_id]['opls_id']}) from {role} atom "
                f"local_id={lid} {oid} "
                f"(had {len(h_nb)} H, expected {expected_h})"
            )

    if not ids_to_remove:
        return corrections

    poly_def["monomer_atoms"] = [a for a in atoms
                                  if a["local_id"] not in ids_to_remove]
    poly_def["intra_bonds"]   = [[i, j] for (i, j) in bonds
                                  if i not in ids_to_remove
                                  and j not in ids_to_remove]
    return corrections

def _auto_fix_alkene_carbon(poly_def: dict, opls_atoms: dict) -> list:
    """Correct alkene carbon (CM) subtype from the number of H actually bonded
    to it, regardless of which CM subtype the formatter chose.

    OPLS-AA splits sp2 alkene carbons by attached-H count:
        CM_1 (R2C=) = 0 H,  CM_2 (RHC=) = 1 H,  CM_3 (H2C=) = 2 H.
    The formatter frequently mislabels a symmetric internal double bond such as
    -CH=CH- (1,4-polybutadiene, polyisoprene backbone): both carbons carry one
    H and must be CM_2, but one is sometimes set to CM_1 (q=0.0), leaving the
    CRU non-neutral (polybutadiene: net -> +0.115 e). This retypes by H count.
    Mirrors _auto_fix_h_types/_auto_fix_valence: the attached-H count is the
    ground truth, so it overrides whatever CM subtype the LLM picked.

    A head/tail alkene carbon spends one sp2 valence on the inter-CRU bond, so
    its attached-H count is taken as-is (the missing substituent is the adjacent
    CRU, not an H), which still yields the correct CM subtype.
    """
    bt_map = {0: "CM_1", 1: "CM_2", 2: "CM_3"}
    atoms = poly_def.get("monomer_atoms", [])
    bonds = poly_def.get("intra_bonds", [])
    id_map = {a["local_id"]: a for a in atoms}
    adj = {}
    for (i, j) in bonds:
        adj.setdefault(i, []).append(j)
        adj.setdefault(j, []).append(i)

    _H_BT = ("H", "HO", "HC", "HA", "HW", "HS")
    def is_H(lid):
        a = id_map.get(lid)
        return bool(a) and opls_atoms.get(a["opls_id"], {}).get("bond_type", "").startswith(_H_BT)

    def bt(lid):
        return opls_atoms.get(id_map[lid]["opls_id"], {}).get("bond_type", "")

    corrections = []
    for a in atoms:
        lid = a["local_id"]
        if bt(lid) != "CM":
            continue
        nH = sum(1 for nb in adj.get(lid, []) if is_H(nb))
        want = bt_map.get(nH)
        if want is None or want not in opls_atoms:
            continue
        if a["opls_id"] != want:
            old = a["opls_id"]
            a["opls_id"] = want
            a["charge"] = opls_atoms[want]["charge"]
            corrections.append(
                f"local_id={lid} {old} -> {want} (alkene C with {nH} attached H)"
            )
    return corrections

def _auto_add_missing_h(poly_def: dict, opls_atoms: dict) -> list:
    """Auto-add missing HC_1 atoms to under-hydrogenated sp3 carbons.

    Counterpart of _auto_fix_valence: instead of removing excess H, this
    adds the required number of HC_1 atoms when a CT carbon has fewer H
    neighbors than its valence allows.  This handles the common LLM failure
    mode of omitting all backbone H atoms and submitting a "heavy-atom only"
    CRU that is always charge-negative.

    New atoms get local_ids starting above the current maximum.
    """
    atoms   = poly_def.get("monomer_atoms", [])
    bonds   = poly_def.get("intra_bonds", [])
    head_id = poly_def.get("head_atom")
    tail_id = poly_def.get("tail_atom")

    if "HC_1" not in opls_atoms:
        return [] 

    hc1_charge = opls_atoms["HC_1"]["charge"]
    max_lid    = max((a["local_id"] for a in atoms), default=0)
    id_map     = {a["local_id"]: a for a in atoms}
    adj        = {}
    for (i, j) in bonds:
        adj.setdefault(i, []).append(j)
        adj.setdefault(j, []).append(i)

    _H_BT = {"H", "HO", "HC", "HA", "HW", "HS"}

    def is_H(lid):
        a = id_map.get(lid)
        return a is not None and any(
            opls_atoms.get(a["opls_id"], {}).get("bond_type", "").startswith(p)
            for p in _H_BT)

    corrections = []
    new_atoms   = []
    new_bonds   = []

    for a in atoms:
        lid = a["local_id"]
        oid = a["opls_id"]
        bt  = opls_atoms.get(oid, {}).get("bond_type", "")
        if not bt.startswith("CT"):
            continue
        neighbors   = adj.get(lid, [])
        h_nb        = [nid for nid in neighbors if is_H(nid)]
        non_h_count = len(neighbors) - len(h_nb)
        inter_bond  = 1 if lid in (head_id, tail_id) else 0
        expected_h  = 4 - non_h_count - inter_bond
        missing     = expected_h - len(h_nb)
        if missing <= 0:
            continue
        for _ in range(missing):
            max_lid += 1
            new_atoms.append({
                "local_id": max_lid,
                "opls_id":  "HC_1",
                "charge":   hc1_charge,
                "comment":  f"HC_1 auto-added to satisfy valence of local_id={lid} {oid}",
            })
            new_bonds.append([lid, max_lid])
            adj.setdefault(lid, []).append(max_lid)
            id_map[max_lid] = new_atoms[-1]
        corrections.append(
            f"local_id={lid} {oid}: added {missing} HC_1 "
            f"(had {len(h_nb)} H, needed {expected_h})"
        )

    if corrections:
        poly_def["monomer_atoms"] = atoms + new_atoms
        poly_def["intra_bonds"]   = bonds + new_bonds

    return corrections



def _auto_fix_o_type(poly_def: dict, opls_atoms: dict) -> list:
    """Replace OS_1 (ether O) with OH_1 (alcohol O) when bonded to HO_1.

    OS_1 = 'ether O (R-O-R)', q=-0.40.  It should NEVER bond to HO_1.
    OH_1 = 'alcohol O (R-OH)',  q=-0.683.  This is the correct hydroxyl oxygen.
    When LLM picks OS_1 for a hydroxyl, auto-correct it before charge check.
    """
    if "OH_1" not in opls_atoms or "OS_1" not in opls_atoms:
        return []

    atoms  = poly_def.get("monomer_atoms", [])
    bonds  = poly_def.get("intra_bonds", [])
    id_map = {a["local_id"]: a for a in atoms}
    adj    = {}
    for (i, j) in bonds:
        adj.setdefault(i, []).append(j)
        adj.setdefault(j, []).append(i)

    corrections = []
    for a in atoms:
        if a["opls_id"] != "OS_1":
            continue
        for nb_id in adj.get(a["local_id"], []):
            nb_a = id_map.get(nb_id)
            if nb_a and nb_a["opls_id"] == "HO_1":
                a["opls_id"] = "OH_1"
                a["charge"]  = opls_atoms["OH_1"]["charge"]
                corrections.append(
                    f"local_id={a['local_id']} OS_1 → OH_1 "
                    f"(bonded to HO_1; ether O cannot bond to alcohol H)"
                )
                break
    return corrections



def _auto_fix_cl_type(poly_def: dict, opls_atoms: dict) -> list:
    """Replace Cl_2 (vinylidene Cl) with Cl_1 (alkyl Cl) when bonded to saturated CT,
    and CT_12 (CH2Cl primary) with CT_2 (alkane CH2) when not actually a CH2Cl.

    Cl_2 (-0.12) is for vinyl/vinylidene C=C; Cl_1 (-0.20) is for saturated C-Cl.
    CT_12 (-0.006) is for primary CH2Cl; in PVC the CH2 is plain alkane → CT_2 (-0.12).
    """
    atoms  = poly_def.get("monomer_atoms", [])
    bonds  = poly_def.get("intra_bonds", [])
    id_map = {a["local_id"]: a for a in atoms}
    adj    = {}
    for (i, j) in bonds:
        adj.setdefault(i, []).append(j)
        adj.setdefault(j, []).append(i)

    corrections = []

    if "Cl_1" in opls_atoms and "Cl_2" in opls_atoms:
        for a in atoms:
            if a["opls_id"] != "Cl_2":
                continue
            for nb_id in adj.get(a["local_id"], []):
                nb_a = id_map.get(nb_id)
                if nb_a and opls_atoms.get(nb_a["opls_id"], {}).get("bond_type", "") == "CT":
                    a["opls_id"] = "Cl_1"
                    a["charge"]  = opls_atoms["Cl_1"]["charge"]
                    corrections.append(
                        f"local_id={a['local_id']} Cl_2 → Cl_1 "
                        f"(bonded to saturated CT; Cl_2 is for C=C vinylidene)"
                    )
                    break

    if "CT_2" in opls_atoms and "CT_12" in opls_atoms:
        for a in atoms:
            if a["opls_id"] != "CT_12":
                continue
            has_cl = any(
                opls_atoms.get(id_map[nb_id]["opls_id"], {}).get("bond_type", "") == "Cl"
                for nb_id in adj.get(a["local_id"], []) if nb_id in id_map)
            if not has_cl:
                a["opls_id"] = "CT_2"
                a["charge"]  = opls_atoms["CT_2"]["charge"]
                corrections.append(
                    f"local_id={a['local_id']} CT_12 → CT_2 "
                    f"(CT_12 is for CH2Cl bonded to Cl; this CH2 has no Cl neighbor)"
                )
    return corrections


def _auto_fix_charge_mismatch(poly_def: dict, opls_atoms: dict,
                               charge_tol: float = 0.01) -> list:
    """Last-resort: swap the atom whose type change minimises |net_charge|.

    When the OPLS table lacks an exact type, no single swap yields net=0.
    This finds the CT atom whose replacement with any same-element CT type
    brings |net_charge| below charge_tol. Only applies when residual
    |net_q| > charge_tol after all other fixes.
    """
    atoms  = poly_def.get("monomer_atoms", [])
    net_q  = round(sum(opls_atoms.get(a["opls_id"], {}).get("charge", 0.0)
                        for a in atoms), 6)
    if abs(net_q) <= charge_tol:
        return []

    ct_types = {k: v for k, v in opls_atoms.items()
                if v.get("bond_type", "") == "CT"}
    best_key  = None
    best_atom = None
    best_net  = abs(net_q)
    for a in atoms:
        oid = a["opls_id"]
        if opls_atoms.get(oid, {}).get("bond_type", "") != "CT":
            continue
        cur_q = opls_atoms[oid]["charge"]
        for ckey, cv in ct_types.items():
            if ckey == oid:
                continue
            new_net = round(net_q - cur_q + cv["charge"], 6)
            if abs(new_net) < best_net:
                best_net  = abs(new_net)
                best_key  = ckey
                best_atom = a
    if best_key is None or best_net >= charge_tol:
        return []

    old_id = best_atom["opls_id"]
    best_atom["opls_id"] = best_key
    best_atom["charge"]  = opls_atoms[best_key]["charge"]
    return [
        f"local_id={best_atom['local_id']} {old_id} → {best_key} "
        f"('{opls_atoms[old_id].get('comment','')}' → "
        f"'{opls_atoms[best_key].get('comment','')}'; "
        f"net {net_q:+.4f} → {best_net:+.4f} e — closest available type)"
    ]

def _renumber_local_ids(poly_def: dict) -> None:
    """Renumber local_ids to be contiguous (1..N) after add/remove fixes.

    The constructor's global_id() maps (chain, monomer, local_id) → atom index
    assuming local_ids run 1..n_mono_atoms with NO gaps. When _auto_fix_valence
    removes an H (e.g. local_id 6), the gap propagates into atom indices and
    bonds point to non-existent atoms → LAMMPS "Bonds assigned incorrectly".
    This compacts local_ids and rewrites intra_bonds / head_atom / tail_atom.
    """
    atoms = poly_def.get("monomer_atoms", [])
    old_ids = [a["local_id"] for a in atoms]
    remap = {old: new for new, old in enumerate(old_ids, start=1)}

    for a in atoms:
        a["local_id"] = remap[a["local_id"]]

    poly_def["intra_bonds"] = [
        [remap[i], remap[j]] for (i, j) in poly_def.get("intra_bonds", [])
        if i in remap and j in remap
    ]
    if poly_def.get("head_atom") in remap:
        poly_def["head_atom"] = remap[poly_def["head_atom"]]
    if poly_def.get("tail_atom") in remap:
        poly_def["tail_atom"] = remap[poly_def["tail_atom"]]



def _auto_fix_head_tail(poly_def: dict, opls_atoms: dict, opls_bonds: dict | None = None) -> list:
    """Validate / correct head_atom and tail_atom (the two inter-CRU linkers).

    A naive "the two heavy atoms of degree 1 are the chain ends" rule is wrong
    for monomers whose PENDANT group ends in a degree-1 heavy atom -- e.g. the
    nitrile N of PAN, a carbonyl O, a halogen. It would pick the pendant tip as
    a chain end and linearise the polymer (and clobber a correct LLM answer).

    Policy:
      1. if the LLM's head/tail already form a real OPLS backbone bond, KEEP them;
      2. otherwise correct to a heavy-degree-1 endpoint pair ONLY if that pair
         forms a valid OPLS inter-monomer bond;
      3. if no confident choice exists, leave head/tail UNTOUCHED so the topology
         validator can reject it and the formatter is asked to fix it -- never
         guess a pendant tip as a backbone end.
    """
    atoms  = poly_def.get("monomer_atoms", [])
    bonds  = poly_def.get("intra_bonds", [])
    id_map = {a["local_id"]: a for a in atoms}
    adj    = {}
    for (i, j) in bonds:
        adj.setdefault(i, []).append(j)
        adj.setdefault(j, []).append(i)

    _H_BT = {"H", "HO", "HC", "HA", "HW", "HS"}
    def bt(lid):
        a = id_map.get(lid)
        return opls_atoms.get(a["opls_id"], {}).get("bond_type", "") if a else ""
    def is_H(lid):
        return any(bt(lid).startswith(p) for p in _H_BT)
    def bond_ok(a_lid, b_lid):
        if opls_bonds is None:
            return None
        return ((bt(a_lid), bt(b_lid)) in opls_bonds or
                (bt(b_lid), bt(a_lid)) in opls_bonds)

    head = poly_def.get("head_atom")
    tail = poly_def.get("tail_atom")

    if (head in id_map and tail in id_map and head != tail
            and not is_H(head) and not is_H(tail)):
        if bond_ok(tail, head) in (True, None):
            return []
    heavy_deg = {a["local_id"]: sum(1 for nb in adj.get(a["local_id"], []) if not is_H(nb))
                 for a in atoms if not is_H(a["local_id"])}
    endpoints = sorted(l for l, d in heavy_deg.items() if d == 1)

    if len(endpoints) == 2:
        e0, e1 = endpoints
        if bond_ok(e0, e1) in (True, None):
            cur = {head, tail}
            if cur == {e0, e1}:
                return []
            poly_def["head_atom"], poly_def["tail_atom"] = e0, e1
            return [f"head/tail corrected to local_ids {e0}/{e1} "
                    f"(backbone ends; was {sorted(x for x in cur if x is not None)})"]
        return [f"head/tail NOT auto-corrected: heavy-degree-1 atoms {endpoints} "
                f"do not form a valid OPLS backbone bond (likely a pendant tip); "
                f"leaving LLM head/tail "
                f"{sorted(x for x in (head, tail) if x is not None)} for validation"]
    return []


def _auto_fix_ether_carbon(poly_def: dict, opls_atoms: dict) -> list:
    """Retype backbone carbons bonded to a TRUE ether O (OS, R-O-R) to the
    ether atom types (CT_5 / HC_3).

    CRITICAL: an ester/carbonate sp3 oxygen ALSO has bond_type "OS". It must
    NOT trigger ether retyping, otherwise an ester alpha carbon (e.g. PLA/PET
    CT_9, q=+0.24) gets demoted to CT_5 (q=+0.14) and the CRU loses neutrality
    (PLA: net -> -0.42 e once the carbonyl carbon is also mis-handled).

    An OS is an ESTER/carbonate oxygen (NOT ether) when the carbon it is bonded
    to is a carbonyl carbon. We detect a carbonyl carbon robustly, WITHOUT
    trusting its assigned atom type, by checking whether that carbon also bears
    a double-bonded carbonyl oxygen (an O-family atom, bond_type starting "O",
    other than the OS itself). This stays correct even when the formatter
    mislabels the carbonyl carbon as CT_1 instead of C_1.
    """
    if "CT_5" not in opls_atoms or "HC_3" not in opls_atoms:
        return []
    atoms   = poly_def.get("monomer_atoms", [])
    bonds   = poly_def.get("intra_bonds", [])
    head_id = poly_def.get("head_atom")
    tail_id = poly_def.get("tail_atom")
    id_map  = {a["local_id"]: a for a in atoms}
    adj     = {}
    for (i, j) in bonds:
        adj.setdefault(i, []).append(j)
        adj.setdefault(j, []).append(i)

    def bt(lid):
        return opls_atoms.get(id_map[lid]["opls_id"], {}).get("bond_type", "")

    def is_carbonyl_carbon(c_lid):
        for nb in adj.get(c_lid, []):
            nbt = bt(nb)
            if nbt == "O" or nbt == "OY":
                return True
        return False

    def is_true_ether_O(o_lid):
        if bt(o_lid) != "OS":
            return False
        for nb in adj.get(o_lid, []):
            if bt(nb).startswith("C") and is_carbonyl_carbon(nb):
                return False
        return True

    ether_O = [a["local_id"] for a in atoms if is_true_ether_O(a["local_id"])]
    if not ether_O:
        return []
    ether_O_set = set(ether_O)

    ether_carbons = set()
    for a in atoms:
        lid = a["local_id"]
        if not bt(lid).startswith("CT"):
            continue
        touches_ether_O = any(nb in ether_O_set for nb in adj.get(lid, []))
        is_endpoint_C   = lid in (head_id, tail_id)
        if touches_ether_O or is_endpoint_C:
            ether_carbons.add(lid)

    corrections = []
    for lid in ether_carbons:
        a = id_map[lid]
        if a["opls_id"] != "CT_5":
            old = a["opls_id"]
            a["opls_id"] = "CT_5"
            a["charge"]  = opls_atoms["CT_5"]["charge"]
            corrections.append(
                f"local_id={lid} {old} -> CT_5 (ether carbon: bonded to / linking ether O)"
            )
        for nb in adj.get(lid, []):
            na = id_map.get(nb)
            if na and bt(nb).startswith("HC") and na["opls_id"] != "HC_3":
                oldh = na["opls_id"]
                na["opls_id"] = "HC_3"
                na["charge"]  = opls_atoms["HC_3"]["charge"]
                corrections.append(
                    f"local_id={nb} {oldh} -> HC_3 (H on ether carbon)"
                )
    return corrections

def _auto_add_missing_ring_h(poly_def: dict, opls_atoms: dict) -> list:
    """Add missing aromatic/alkene hydrogens to under-coordinated sp2 carbons.

    Counterpart of _auto_add_missing_h, which only fills sp3 CT carbons. Aromatic
    CA carbons (3-coordinate, ring H = HA) and alkene CM carbons (3-coordinate,
    vinylic H) are otherwise left under-hydrogenated when the LLM omits ring/vinyl
    H -- the classic polystyrene failure that leaves the CRU at a large negative
    net charge (e.g. -0.46 e for a phenyl ring missing its 5 aromatic H).
    """
    atoms   = poly_def.get("monomer_atoms", [])
    bonds   = poly_def.get("intra_bonds", [])
    head_id = poly_def.get("head_atom")
    tail_id = poly_def.get("tail_atom")

    ha_key = next((k for k, v in opls_atoms.items() if v.get("bond_type") == "HA"), None)
    ch_key = next((k for k, v in opls_atoms.items()
                   if v.get("bond_type") == "HC" and abs(v.get("charge", 0.0) - 0.115) < 0.01), None)
    targets = {"CA": (3, ha_key), "CM": (3, ch_key)}

    def _n_h_key(n_opls):
        ncmt = opls_atoms.get(n_opls, {}).get("comment", "").lower()
        for desc in ("primary amide", "secondary amide", "imide",
                     "secondary amine", "primary amine"):
            if desc in ncmt:
                for k, v in opls_atoms.items():
                    vc = v.get("comment", "").lower()
                    if v.get("bond_type", "").startswith("H") and "h(n)" in vc and desc in vc:
                        return k
        return None

    max_lid = max((a["local_id"] for a in atoms), default=0)
    id_map  = {a["local_id"]: a for a in atoms}
    adj = {}
    for (i, j) in bonds:
        adj.setdefault(i, []).append(j)
        adj.setdefault(j, []).append(i)

    _H_BT = {"H", "HO", "HC", "HA", "HW", "HS"}
    def is_H(lid):
        a = id_map.get(lid)
        return a is not None and any(
            opls_atoms.get(a["opls_id"], {}).get("bond_type", "").startswith(p) for p in _H_BT)

    corrections, new_atoms, new_bonds = [], [], []
    for a in atoms:
        lid = a["local_id"]
        bt  = opls_atoms.get(a["opls_id"], {}).get("bond_type", "")
        if bt in ("N", "NT"):
            exp_total, hkey = 3, _n_h_key(a["opls_id"])
        elif bt in targets:
            exp_total, hkey = targets[bt]
        else:
            continue
        if hkey is None:
            continue
        nbrs   = adj.get(lid, [])
        h_nb   = [n for n in nbrs if is_H(n)]
        non_h  = len(nbrs) - len(h_nb)
        inter  = 1 if lid in (head_id, tail_id) else 0
        missing = exp_total - non_h - inter - len(h_nb)
        for _ in range(missing):
            max_lid += 1
            new_atoms.append({
                "local_id": max_lid,
                "opls_id":  hkey,
                "charge":   opls_atoms[hkey]["charge"],
                "comment":  f"{hkey} auto-added to satisfy valence of local_id={lid} {a['opls_id']}",
            })
            new_bonds.append([lid, max_lid])
            corrections.append(f"local_id={lid} {a['opls_id']}: added {hkey} ({bt} H)")
    if new_atoms:
        poly_def["monomer_atoms"].extend(new_atoms)
        poly_def["intra_bonds"].extend(new_bonds)
    return corrections


def _auto_neutralize_residual(poly_def: dict, opls_atoms: dict, tol: float = 0.01,
                             max_spread: float = 0.25) -> list:
    """Last-resort neutraliser for CRUs that no table type-swap can zero out
    (e.g. secondary-amide polymers such as nylon-6, whose OPLS amide group sums
    to -0.20 e and for which this table has no amide-alpha-carbon charge). Pins
    explicit charges to the table, then spreads any small residual over the heavy
    atoms so the repeat unit is exactly neutral for the periodic simulation. Only
    fires for a small residual (<= max_spread); a larger imbalance still signals a
    real typing error and is left for the retry loop to flag."""
    atoms = poly_def.get("monomer_atoms", [])
    if not atoms:
        return []
    for a in atoms:
        a["charge"] = opls_atoms.get(a["opls_id"], {}).get("charge", a.get("charge", 0.0))
    net = round(sum(a["charge"] for a in atoms), 6)
    if abs(net) <= tol or abs(net) > max_spread:
        return []
    heavy = [a for a in atoms
             if not opls_atoms.get(a["opls_id"], {}).get("bond_type", "").startswith("H")]
    targetset = heavy or atoms
    shift = net / len(targetset)
    for a in targetset:
        a["charge"] = round(a["charge"] - shift, 6)
    return [f"spread residual net {net:+.4f} e over {len(targetset)} heavy atoms "
            f"({-shift:+.5f} e each) to enforce exact neutrality "
            f"(table lacks an exact-neutral typing for this CRU)"]


def _apply_all_autofixes(poly_def: dict, opls: dict, charge_tol: float = 0.01) -> None:
    """Run the full ordered auto-fix suite on poly_def in place, printing each
    correction. Shared by the homopolymer and copolymer definer loops so BOTH
    paths get identical repair capability. (The copolymer loop previously ran
    only _auto_fix_h_types, leaving charge-sensitive components such as
    polystyrene stuck at a non-zero net charge and looping forever.)"""
    for fixes, tag in (
        (_auto_fix_h_types(poly_def, opls["atom"]),                      "Auto-fix H"),
        (_auto_fix_o_type(poly_def, opls["atom"]),                       "Auto-fix O"),
        (_auto_fix_cl_type(poly_def, opls["atom"]),                      "Auto-fix Cl"),
        (_auto_fix_head_tail(poly_def, opls["atom"], opls["bond"]),      "Auto-fix head/tail"),
        (_auto_fix_ether_carbon(poly_def, opls["atom"]),                 "Auto-fix ether"),
        (_auto_fix_valence(poly_def, opls["atom"]),                      "Auto-fix valence"),
        (_auto_add_missing_h(poly_def, opls["atom"]),                    "Auto-add H"),
        (_auto_add_missing_ring_h(poly_def, opls["atom"]),               "Auto-add ring/vinyl H"),
        (_auto_fix_alkene_carbon(poly_def, opls["atom"]),                "Auto-fix alkene"),
        (_auto_fix_charge_mismatch(poly_def, opls["atom"], charge_tol), "Auto-fix charge"),
        (_auto_neutralize_residual(poly_def, opls["atom"], charge_tol),  "Auto-neutralize residual"),
    ):
        if fixes:
            for _f in fixes:
                print(f"  [{tag}] {_f}")


def _canonical_copoly_poly_def(component_defs: list, state: dict) -> dict:
    """Combine per-component poly_defs into ONE canonical copolymer poly_def
    (monomer_types + sequence). The copolymer definer otherwise leaves
    state["poly_def"] as only the first component, so every downstream stage
    that reads state["poly_def"] (bead mapping, CG construction) would see a
    styrene-only homopolymer and never map/bond the butadiene beads. The block
    sequence is regenerated with the SAME logic the AA constructor used
    (copolymer.generate_sequence) so bead.data and the AA structure agree."""
    import numpy as np
    labels = [chr(ord("A") + i) for i in range(len(component_defs))]
    monomer_types = {}
    for lbl, pd in zip(labels, component_defs):
        monomer_types[lbl] = {
            "name":          pd.get("polymer_name", lbl),
            "monomer_atoms": pd["monomer_atoms"],
            "intra_bonds":   pd.get("intra_bonds", []),
            "head_atom":     pd["head_atom"],
            "tail_atom":     pd["tail_atom"],
            "n_atoms":       len(pd["monomer_atoms"]),
        }
    spec_dict = state.get("copolymer_spec") or {}
    n_mono    = (state.get("sim_params") or {}).get("n_monomers", 10)
    try:
        from copolymer import CopolymerSpec, generate_sequence as _copoly_seq
        _fields  = CopolymerSpec.__dataclass_fields__
        spec     = CopolymerSpec(**{k: v for k, v in spec_dict.items() if k in _fields})
        idx_seq  = _copoly_seq(n_mono, spec, np.random.default_rng(0))
        sequence = [labels[i % len(labels)] for i in idx_seq]
    except Exception:
        sequence = [labels[(i * len(labels)) // max(1, n_mono)] for i in range(n_mono)]
    return {
        "polymer_name":  state.get("polymer_name", "copolymer"),
        "polymer_type":  "copolymer",
        "sequence_type": spec_dict.get("architecture", "block"),
        "monomer_types": monomer_types,
        "sequence":      sequence,
    }


def definer_agent(state: LAMMPSState) -> LAMMPSState:
    from aa_constructor import load_opls
    opls       = load_opls(OPLS_DIR)
    opls_table = _format_opls_table(opls["atom"])

    _MAX_CHARGE_RETRIES = 5
    _CHARGE_TOL         = 0.01

    base_system = DEFINER_SYSTEM.format(opls_table=opls_table)

    if not state.get("is_copolymer", False):
        cur_name     = state["polymer_name"]
        cur_poly_def = state.get("poly_def")
        if cur_poly_def is None:
            print("\n" + "="*60)
            print("[Formatter] Generating topology for: " + cur_name)
            print("="*60)
            user_content = "Generate the polymer topology for: " + cur_name
        else:
            print("\n" + "="*60)
            print("[Formatter] Revising... (revision " + str(state["revision_count"]) + ")")
            print("Feedback:\n" + str(state["review_feedback"]))
            print("="*60)
            user_content = (
                "Original polymer: " + cur_name + "\n\n"
                "Previous topology JSON:\n" + json.dumps(cur_poly_def, indent=2) + "\n\n"
                "Reviewer feedback (fix these issues):\n" + str(state["review_feedback"]) + "\n\n"
                "Generate a corrected polymer topology JSON."
            )
        _expected_formula = get_expected_cru_formula(cur_name, llm)
        if _expected_formula:
            print(f"[Formatter] Expected repeat-unit formula for '{cur_name}': "
                  f"{_expected_formula}")
        else:
            print("[Formatter] No independent formula available; "
                  "the composition cross-check will be skipped.")

        charge_feedback = None
        full_response   = ""
        for attempt in range(_MAX_CHARGE_RETRIES):
            if charge_feedback is None:
                messages = [SystemMessage(content=base_system), HumanMessage(content=user_content)]
            else:
                messages = [SystemMessage(content=base_system), HumanMessage(content=user_content),
                            AIMessage(content=full_response),
                            HumanMessage(content=(f"CORRECTION NEEDED (attempt {attempt}/{_MAX_CHARGE_RETRIES}):\n"
                                                   f"{charge_feedback}\nReturn corrected JSON only."))]
            if attempt > 0:
                print(f"\n[Formatter] Retry {attempt}/{_MAX_CHARGE_RETRIES}...")
            print("\n--- Formatter output ---")
            full_response = ""
            for chunk in llm.stream(messages):
                print(chunk.content, end="", flush=True)
                full_response += chunk.content
            print("\n--- Formatter done ---")
            clean = re.sub(r'^```(?:json)?\s*', '', full_response.strip())
            clean = re.sub(r'```\s*$', '', clean).strip()
            try:
                poly_def = json.loads(clean)
            except json.JSONDecodeError as e:
                if attempt == _MAX_CHARGE_RETRIES - 1:
                    raise ValueError("Formatter returned invalid JSON: " + str(e) + "\n\nRaw:\n" + full_response)
                charge_feedback = f"JSON parse error: {e}. Return valid JSON only."
                continue
            bad_ids = []
            for atom in poly_def.get("monomer_atoms", []):
                oid = atom.get("opls_id")
                if oid not in opls["atom"]: bad_ids.append(oid)
                else: atom["charge"] = opls["atom"][oid]["charge"]
            if bad_ids:
                if attempt == _MAX_CHARGE_RETRIES - 1: raise ValueError(f"Unknown opls_id(s): {bad_ids}")
                charge_feedback = f"Unknown opls_id(s): {bad_ids}. Use only keys from the table."
                continue
            _apply_all_autofixes(poly_def, opls, _CHARGE_TOL)
            net_q    = sum(a.get("charge", opls["atom"][a["opls_id"]]["charge"])
                           for a in poly_def.get("monomer_atoms", []))
            expected = poly_def.get("expected_net_charge", 0.0)
            if abs(net_q - expected) <= _CHARGE_TOL:
                _renumber_local_ids(poly_def)  
                _topo_ok, _topo_errs, _topo_warns = validate_monomer_topology(poly_def, opls)
                for _w in _topo_warns: print(f"  [topology] {_w}")
                if _topo_ok:
                    _fchk = compare_cru_formula(poly_def, opls["atom"], _expected_formula)
                    if _fchk["checked"] and not _fchk["ok"]:
                        print(f"[Formatter] Formula mismatch (attempt {attempt+1}): "
                              f"{_fchk['message']} -> retrying")
                        charge_feedback = (
                            "The repeat unit is charge-neutral and has valid valence, but it is "
                            "the WRONG MOLECULE.\n" + _fchk["message"] + "\n"
                            "This almost always means a substituent was dropped or added on a "
                            "backbone carbon. Re-derive the repeat unit for '" + cur_name + "' "
                            "from its structure, count the substituents on every backbone carbon, "
                            "and make the atom list match the expected formula exactly. "
                            "Remember a carbon with two alkyl branches is quaternary (no H).")
                        continue
                    _fnote = (f"  |  formula {_fchk['built']} ({_fchk['mass']:.2f} amu)"
                              + (" == expected" if _fchk["checked"] else " (unverified)"))
                    print(f"[Formatter] Charge OK: net = {net_q:+.4f} e  |  topology OK{_fnote}")
                    return {**state, "poly_def": poly_def, "status": "constructing"}
                charge_feedback = (
                    "The molecule is charge-neutral but its TOPOLOGY is chemically invalid. "
                    "Fix the monomer connectivity (intra_bonds / head_atom / tail_atom): every "
                    "bond must be a real chemical bond and every atom must have correct valence. "
                    "Pendant groups (nitrile C#N, phenyl, ester, etc.) must BRANCH OFF the backbone, "
                    "never be inserted INTO it; head_atom and tail_atom must be the two backbone "
                    "atoms that link to the neighbouring repeat units.\nIssues:\n  - "
                    + "\n  - ".join(_topo_errs))
                print(f"[Formatter] Topology error (attempt {attempt+1}): "
                      f"{len(_topo_errs)} issue(s) -> retrying")
                continue
            atom_lines = [f"  local_id={a['local_id']} opls_id={a['opls_id']} q={opls['atom'][a['opls_id']]['charge']:+.4f} ({opls['atom'][a['opls_id']].get('comment','')})"
                          for a in poly_def.get("monomer_atoms", [])]
            charge_feedback = (f"net charge = {net_q:+.4f} e (expected {expected:+.4f}). "
                               f"Per-atom charges:\n" + "\n".join(atom_lines))
            print(f"[Formatter] Charge error (attempt {attempt+1}): net = {net_q:+.4f} e  -> retrying")
        raise ValueError(f"definer_agent failed after {_MAX_CHARGE_RETRIES} attempts. "
                         f"Last net charge = {net_q:+.4f} e.\n{charge_feedback}")

    components = state.get("_copoly_components") or [state["polymer_name"]]
    cur_idx    = state.get("_copoly_current_idx") or 0
    cur_name   = components[cur_idx]

    existing_defs = state.get("poly_defs") or []
    cur_poly_def  = existing_defs[cur_idx] if len(existing_defs) > cur_idx else None

    is_revision = (cur_poly_def is not None and
                   state.get("review_feedback") and
                   state.get("revision_count", 0) > 0)

    if not is_revision:
        n_total = len(components)
        print("\n" + "="*60)
        print(f"[Formatter] Generating topology for component {cur_idx+1}/{n_total}: {cur_name}")
        print("="*60)
        user_content = "Generate the polymer topology for: " + cur_name
    else:
        print("\n" + "="*60)
        print(f"[Formatter] Revising component {cur_idx+1} (revision {state['revision_count']})")
        print("Feedback:\n" + str(state["review_feedback"]))
        print("="*60)
        user_content = (
            "Original polymer: " + cur_name + "\n\n"
            "Previous topology JSON:\n" + json.dumps(cur_poly_def, indent=2) + "\n\n"
            "Reviewer feedback (fix these issues):\n" + str(state["review_feedback"]) + "\n\n"
            "Generate a corrected polymer topology JSON."
        )

    charge_feedback = None
    full_response   = ""

    for attempt in range(_MAX_CHARGE_RETRIES):
        if charge_feedback is None:
            messages = [
                SystemMessage(content=base_system),
                HumanMessage(content=user_content),
            ]
        else:
            messages = [
                SystemMessage(content=base_system),
                HumanMessage(content=user_content),
                AIMessage(content=full_response),
                HumanMessage(content=(
                    f"CORRECTION NEEDED (attempt {attempt}/{_MAX_CHARGE_RETRIES}):\n"
                    f"{charge_feedback}\nReturn corrected JSON only."
                )),
            ]

        if attempt > 0:
            print(f"\n[Formatter] Retry {attempt}/{_MAX_CHARGE_RETRIES}...")

        print("\n--- Formatter output ---")
        full_response = ""
        for chunk in llm.stream(messages):
            print(chunk.content, end="", flush=True)
            full_response += chunk.content
        print("\n--- Formatter done ---")

        clean = re.sub(r'^```(?:json)?\s*', '', full_response.strip())
        clean = re.sub(r'```\s*$', '', clean).strip()
        try:
            poly_def = json.loads(clean)
        except json.JSONDecodeError as e:
            if attempt == _MAX_CHARGE_RETRIES - 1:
                raise ValueError("Formatter returned invalid JSON: " + str(e) + "\n\nRaw:\n" + full_response)
            charge_feedback = f"JSON parse error: {e}. Return valid JSON only."
            continue

        bad_ids = []
        for atom in poly_def.get("monomer_atoms", []):
            oid = atom.get("opls_id")
            if oid not in opls["atom"]:
                bad_ids.append(oid)
            else:
                atom["charge"] = opls["atom"][oid]["charge"]

        if bad_ids:
            if attempt == _MAX_CHARGE_RETRIES - 1:
                raise ValueError(f"Unknown opls_id(s): {bad_ids}")
            charge_feedback = f"Unknown opls_id(s): {bad_ids}. Use only keys from the table."
            continue

        _apply_all_autofixes(poly_def, opls, _CHARGE_TOL)
        net_q    = sum(a.get("charge", opls["atom"][a["opls_id"]]["charge"])
                       for a in poly_def.get("monomer_atoms", []))
        expected = poly_def.get("expected_net_charge", 0.0)

        if abs(net_q - expected) <= _CHARGE_TOL:
            _topo_ok, _topo_errs, _topo_warns = validate_monomer_topology(poly_def, opls)
            for _w in _topo_warns: print(f"  [topology] {_w}")
            if not _topo_ok:
                charge_feedback = (
                    "The component is charge-neutral but its TOPOLOGY is chemically invalid. "
                    "Fix intra_bonds / head_atom / tail_atom: every bond must be a real chemical "
                    "bond with correct valence; pendant groups must BRANCH OFF the backbone, not "
                    "be inserted INTO it; head_atom and tail_atom must be the two backbone atoms.\n"
                    "Issues:\n  - " + "\n  - ".join(_topo_errs))
                print(f"[Formatter] Topology error (attempt {attempt+1}): "
                      f"{len(_topo_errs)} issue(s) -> retrying")
                continue
            print(f"[Formatter] Charge OK: net = {net_q:+.4f} e  |  topology OK")

            new_defs = list(existing_defs)
            if len(new_defs) > cur_idx:
                new_defs[cur_idx] = poly_def
            else:
                new_defs.append(poly_def)

            if cur_idx + 1 >= len(components):
                return {**state,
                        "poly_defs":           new_defs,
                        "poly_def":            _canonical_copoly_poly_def(new_defs, state),
                        "_copoly_current_idx": cur_idx,
                        "status":              "constructing"}
            else:
                return {**state,
                        "poly_defs":           new_defs,
                        "_copoly_current_idx": cur_idx + 1,
                        "status":              "defining_next_component"}

        charge_feedback = _build_charge_feedback(poly_def, opls, net_q, expected)
        print(f"[Formatter] Charge error (attempt {attempt+1}): net = {net_q:+.4f} e  -> retrying")

    if is_revision and cur_poly_def is not None:
        print(f"[Formatter] Revision of component {cur_idx+1} could not reach "
              f"neutrality (last net = {net_q:+.4f} e); keeping the previously "
              f"validated topology and continuing.")
        new_defs = list(existing_defs)
        if len(new_defs) > cur_idx:
            new_defs[cur_idx] = cur_poly_def
        else:
            new_defs.append(cur_poly_def)
        if cur_idx + 1 >= len(components):
            return {**state,
                    "poly_defs":           new_defs,
                    "poly_def":            _canonical_copoly_poly_def(new_defs, state),
                    "_copoly_current_idx": cur_idx,
                    "status":              "constructing"}
        else:
            return {**state,
                    "poly_defs":           new_defs,
                    "_copoly_current_idx": cur_idx + 1,
                    "status":              "defining_next_component"}

    raise ValueError(
        f"definer_agent failed to achieve charge neutrality after {_MAX_CHARGE_RETRIES} attempts. "
        f"Last net charge = {net_q:+.4f} e.\n"
        f"Last charge breakdown:\n{charge_feedback}"
    )


In [ ]:
# Param confirm node  --  show params once, get user approval

AVOGADRO = 6.02214076e23

def _monomer_mass(poly_def: dict, opls_atoms: dict) -> float:
    return sum(opls_atoms[a["opls_id"]]["mass"] for a in poly_def["monomer_atoms"])


def _avg_monomer_mass_from_state(state: dict, opls_atoms: dict) -> float:
    """Average monomer mass weighted by copolymer fractions (or single CRU mass)."""
    poly_defs = state.get("poly_defs") or [state["poly_def"]]
    masses    = [_monomer_mass(pd, opls_atoms) for pd in poly_defs if pd]
    if len(masses) == 1:
        return masses[0]
    spec   = state.get("copolymer_spec") or {}
    fracs  = spec.get("fractions", [1.0 / len(masses)] * len(masses))
    fracs  = fracs[:len(masses)]
    total  = sum(fracs)
    fracs  = [f / total for f in fracs]
    return sum(m * f for m, f in zip(masses, fracs))

def _density_from_box(box_size: float, n_chains: int,
                      n_monomers: int, monomer_mass: float) -> float:
    V_cm3 = (box_size * 1e-8) ** 3
    return (n_chains * n_monomers * monomer_mass) / (AVOGADRO * V_cm3)


def aa_params_node(state: LAMMPSState) -> LAMMPSState:
    sd = state["session_dir"]

    opls = load_opls(OPLS_DIR)
    if not state.get("is_copolymer", False):
        poly_def = state["poly_def"]
        m_mass   = _monomer_mass(poly_def, opls["atom"])
    else:
        poly_def = (state.get("poly_defs") or [state["poly_def"]])[0]
        m_mass   = _avg_monomer_mass_from_state(state, opls["atom"])

    prev       = state.get("sim_params") or {}
    n_chains   = prev.get("n_chains",   DEFAULT_N_CHAINS)
    n_monomers = prev.get("n_monomers", DEFAULT_N_MONOMERS)
    density    = prev.get("density",    DEFAULT_DENSITY)
    box_size   = round(calculate_box_size(n_chains, n_monomers, m_mass, density), 3)

    params = {
        "n_chains": n_chains, "n_monomers": n_monomers,
        "density": density, "box_size": box_size,
        "monomer_mass": round(m_mass, 3),
    }

    msg = (
        "chain length: " + str(n_monomers) + ", "
        "number of chains: " + str(n_chains) + ", "
        "density: " + str(density) + " g/cm\u00b3, "
        "box size: " + str(box_size) + " \u00c5. Confirm?"
    )
    if _log_once(sd, "cgmas", msg):   
        print("\n" + "="*60)
        print("CGMas: " + msg)
        print("="*60)

    user_reply = _resume(state, "aa_params", auto_reply=_auto_param_reply(state, "aa_params"))
    _log(sd, "user", user_reply)

    _cp = json.dumps(params)
    system_content = (
        "Parse the user reply about simulation parameters."
        " Current params: " + _cp + "."
        " If user approves (ok/yes/go on/confirm) -> APPROVED."
        " Otherwise ONLY JSON with changed fields."
        " Respond ONLY APPROVED or JSON. No other text."
    )
    parse_resp = llm.invoke([
        SystemMessage(content=system_content),
        HumanMessage(content=user_reply),
    ])
    parsed = parse_resp.content.strip()

    if parsed == "APPROVED":
        reply = "Confirmed. Proceeding to data file construction..."
        print("\nCGMas: " + reply)
        _log(sd, "cgmas", reply)
        return {**state, "sim_params": params, "status": "params_confirmed"}

    try:
        changes = json.loads(re.sub(r'^```(?:json)?\s*|```\s*$', '', parsed).strip())
        updated = {**params, **changes}
        if "box_size" in changes and "density" not in changes:
            updated["density"] = round(
                _density_from_box(updated["box_size"], updated["n_chains"],
                                   updated["n_monomers"], m_mass), 4)
        elif "box_size" not in changes:
            updated["box_size"] = round(
                calculate_box_size(updated["n_chains"], updated["n_monomers"],
                                   m_mass, updated["density"]), 3)
    except Exception:
        updated = params

    return {**state, "sim_params": updated, "status": ("params_confirmed" if state.get("auto") else "params_pending")}


In [ ]:
# Constructor node  --  constructor.py tool call

def aa_constructor_node(state: LAMMPSState) -> LAMMPSState:
    sd = state["session_dir"]
    print("\n" + "="*60)
    print("[Constructor] Generating data file...")
    print("="*60)

    params        = state["sim_params"]
    poly_name     = re.sub(r"[^\w]+", "_", state.get("polymer_name", "polymer").lower())
    datafile_path = os.path.join(sd, poly_name + "_ini.data")

    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        if not state.get("is_copolymer", False):
            build_and_write(
                state["poly_def"], OPLS_DIR, datafile_path,
                n_chains   = params["n_chains"],
                n_monomers = params["n_monomers"],
                density    = params["density"],
                seed       = DEFAULT_SEED,
            )
        else:
            spec_dict = state.get("copolymer_spec") or {"architecture": "random", "fractions": []}
            spec      = CopolymerSpec(**{k: v for k, v in spec_dict.items()
                                          if k in CopolymerSpec.__dataclass_fields__})
            build_and_write_copoly(
                poly_defs   = state.get("poly_defs") or [state["poly_def"]],
                opls_dir    = OPLS_DIR,
                output_path = datafile_path,
                spec        = spec,
                n_chains    = params["n_chains"],
                n_monomers  = params["n_monomers"],
                density     = params["density"],
                seed        = DEFAULT_SEED,
            )

    seen = set()
    for w in caught:
        msg = str(w.message)
        if msg not in seen:
            print("  warning: " + msg)
            seen.add(msg)

    print("\n[Constructor] File written: " + datafile_path)
    return {**state, "datafile_path": datafile_path, "status": "reviewing"}


In [10]:
# Reviewer agent

AA_REVIEW_SYSTEM = """You are a senior computational materials scientist reviewing LAMMPS data files
generated for amorphous polymer simulations (OPLS-AA, full atom style).

[Format checks]
1. Header counts match data sections.
2. Full atom style: atom-ID mol-ID atom-type q x y z ix iy iz.
3. All declared sections present and non-empty.

[Force-field checks]
4. Coefficients present for ALL declared types.
5. LJ epsilon (kcal/mol) and sigma (A) physically reasonable.
6. Bond/angle/dihedral types consistent with topology.

[Chemistry / topology checks]
7. monomer_atoms JSON makes chemical sense for the requested polymer.
8. head_atom / tail_atom consistent with inter-monomer connectivity.

[Request fulfillment]
9. The topology represents what the user asked for.

[ALREADY-VERIFIED FACTS — DO NOT RE-LITIGATE]
The following were established upstream and are GUARANTEED true by the time you
see this file. Treat them as settled and NEVER raise them as issues:
  A. Per-repeat-unit charge neutrality. The topology only reaches you AFTER the
     Formatter confirmed the CRU net charge is 0 within tolerance. Do NOT ask to
     "confirm charge is ~0", do NOT recompute per-CRU charge, and do NOT request
     the replication mapping to verify charge. Charge neutrality is DONE.
  B. File completeness. The Constructor wrote the ENTIRE data file and reported
     the section counts; the file is complete and all sections are present and
     non-empty. The excerpt you receive is INTENTIONALLY TRUNCATED to the first
     3000 characters for brevity — truncation of the PREVIEW is NOT truncation
     of the file. Do NOT raise "file looks truncated", "cannot confirm all
     sections present", or "please provide the remaining sections".

[Rules for issuing REVISION_NEEDED]
- Only flag a CONCRETE, DEMONSTRABLE defect you can point to in the JSON or in
  the visible excerpt (e.g. a wrong atom type, an impossible bond, a missing
  declared Coeff for a type that appears).
- NEVER issue REVISION_NEEDED for "insufficient information", "cannot verify
  from the excerpt", "please confirm", or "please provide more of the file".
  Missing visibility is NOT a defect — if you cannot see a problem, there is
  none to report; APPROVE.
- Do not re-open items A or B above under any phrasing.
- When in doubt, APPROVE.

Respond EXACTLY:
STATUS: APPROVED
FEEDBACK: Looks good.

OR

STATUS: REVISION_NEEDED
FEEDBACK:
1. <issue>
..."""


def aa_review_agent(state: LAMMPSState) -> LAMMPSState:
    print("\n" + "="*60)
    print("[Reviewer] Reviewing data file...")
    print("="*60)

    with open(state["datafile_path"]) as f:
        preview = f.read(3000)

    messages = [
        SystemMessage(content=AA_REVIEW_SYSTEM),
        HumanMessage(content=(
            "User request: " + str(state["polymer_name"]) + "\n\n"
            "Polymer topology JSON:\n" + json.dumps(state["poly_def"], indent=2) + "\n\n"
            "LAMMPS data file (first 3000 chars):\n" + preview + "\n[...]"
        ))
    ]

    print("\n--- Reviewer output ---")
    full_response = ""
    for chunk in llm.stream(messages):
        print(chunk.content, end="", flush=True)
        full_response += chunk.content
    print("\n--- Reviewer done ---")

    if "STATUS: APPROVED" in full_response:
        print("\n[Reviewer] -> APPROVED")
        return {
            **state,
            "review_feedback":     full_response,
            "final_datafile_path": state["datafile_path"],
            "status": "approved",
        }
    else:
        feedback = full_response.split("FEEDBACK:", 1)[-1].strip() if "FEEDBACK:" in full_response else full_response
        print("\n[Reviewer] -> REVISION NEEDED")
        return {**state, "review_feedback": feedback, "status": "reviewing"}


In [ ]:
# Simulation agent nodes + routing functions

def sim_params_node(state: LAMMPSState) -> LAMMPSState:
    sd = state["session_dir"]

    prev     = state.get("sim_script_params") or {}
    t_init   = prev.get("t_init",   DEFAULT_T_INIT)
    t_max    = prev.get("t_max",    DEFAULT_T_MAX)
    n_anneal = prev.get("n_anneal", DEFAULT_N_ANNEAL)
    equil_ps = prev.get("equil_ps", DEFAULT_EQUIL_PS)
    params   = {"t_init": t_init, "t_max": t_max,
                "n_anneal": n_anneal, "equil_ps": equil_ps}

    params   = _apply_temp_directives(state, "sim_params", params)
    t_init, t_max, n_anneal = params["t_init"], params["t_max"], params["n_anneal"]

    cycle_word = "cycles" if n_anneal > 1 else "cycle"
    msg = (
        "Operating temp.: " + str(t_init) + " K, "
        "Up temp.: " + str(t_max) + " K, "
        "annealing: " + str(n_anneal) + " " + cycle_word + ", "
        "Total equilibrium time: " + str(equil_ps) + " ps. Confirm?"
    )
    if _log_once(sd, "cgmas", msg):   
        print("\n" + "="*60)
        print("CGMas: " + msg)
        print("="*60)

    user_reply = _resume(state, "sim_params", auto_reply=_auto_param_reply(state, "sim_params"))
    _log(sd, "user", user_reply)

    _cp2 = json.dumps(params)
    system_content = (
        "Parse the user reply about LAMMPS simulation parameters."
        " Current params: " + _cp2 + "."
        " If user approves (ok/yes/go on/confirm) -> APPROVED."
        " Otherwise ONLY JSON with changed fields."
        " e.g. max temp 450K -> t_max=450, 2ns eq -> equil_ps=2000."
        " Respond ONLY APPROVED or JSON. No other text."
    )
    parse_resp = llm.invoke([
        SystemMessage(content=system_content),
        HumanMessage(content=user_reply),
    ])
    parsed = parse_resp.content.strip()

    if parsed == "APPROVED":
        reply = "Confirmed. Generating simulation script..."
        print("\nCGMas: " + reply)
        _log(sd, "cgmas", reply)
        return {**state, "sim_script_params": params, "status": "sim_confirmed"}

    try:
        changes = json.loads(re.sub(r'^```(?:json)?\s*|```\s*$', '', parsed).strip())
        updated = {**params, **changes}
    except Exception:
        updated = params
    updated = _apply_temp_directives(state, "sim_params", updated)

    return {**state, "sim_script_params": updated, "status": ("sim_confirmed" if state.get("auto") else "sim_pending")}

def sim_build_node(state: LAMMPSState) -> LAMMPSState:
    sd = state["session_dir"]
    print("\n" + "="*60)
    print("[SimConstructor] Generating simulation script...")
    print("="*60)

    params       = state["sim_script_params"]
    datafile     = state["final_datafile_path"] or state["datafile_path"]
    poly_name    = re.sub(r"[^\w]+", "_", state.get("polymer_name", "polymer").lower())
    in_file_path = os.path.join(sd, "in." + poly_name + "_AAEQ")

    build_in_file(
        template_path = TEMPLATE_IN,
        datafile_path = datafile,
        output_path   = in_file_path,
        t_init        = params["t_init"],
        t_max         = params["t_max"],
        n_anneal      = params["n_anneal"],
        equil_ps      = params["equil_ps"],
    )
    print(f"[SimConstructor] AA temperatures written to {os.path.basename(in_file_path)}: "
          f"Tini = {params['t_init']} K, Tfin = {params['t_max']} K")

    reply   = "Script file construction complete. (" + os.path.basename(in_file_path) + ")"
    run_msg = "Ready to simulate. Type 'OK. please simulate.' to start LAMMPS."
    print("\nCGMas: " + reply)
    print("CGMas: " + run_msg)
    _log(sd, "cgmas", reply)
    _log(sd, "cgmas", run_msg)

    return {**state, "in_file_path": in_file_path, "status": "script_ready"}

def aa_run_node(state: LAMMPSState) -> LAMMPSState:
    sd = state["session_dir"]

    user_reply = _resume(state, "aa_run", auto_reply="yes, please run the LAMMPS simulation now")
    _log(sd, "user", user_reply)

    resp = llm.invoke([
        SystemMessage(content=(
            "Does the user want to start LAMMPS simulation? "
            "If yes (simulate/run/go/start/ok/please simulate) respond: YES. "
            "Otherwise respond: NO. Only YES or NO."
        )),
        HumanMessage(content=user_reply),
    ])
    wants_run = "YES" in resp.content.upper()

    if not wants_run:
        msg = "Simulation not started. Let me know when you're ready."
        print("\nCGMas: " + msg)
        _log(sd, "cgmas", msg)
        return {**state, "status": "waiting_run"}

    print("\n" + "="*60)
    print("[LAMMPS] Starting simulation...")
    print("="*60)

    result = run_lammps(
        in_file    = state["in_file_path"],
        datafile   = state["final_datafile_path"] or state["datafile_path"],
        output_dir = sd,
        n_cores    = 12,
        label      = "aa",
    )

    if result["returncode"] == 0:
        reply = (
            "Simulation done! Files saved in " + os.path.basename(sd) + "/  "
            "-- Log: " + os.path.basename(result["log_path"]) + ", "
            "Screen: " + os.path.basename(result["screen_path"])
        )
        import glob as _glob
        fin_data_cands = (_glob.glob(os.path.join(sd, "AAEQfin.data")) or
                          _glob.glob(os.path.join(sd, "*fin*.data")))
        aa_fin_path = fin_data_cands[0] if fin_data_cands else None
    else:
        reply = "LAMMPS error (code " + str(result["returncode"]) + "). " + result.get("stderr", "")[-300:]
        aa_fin_path = None

    print("\nCGMas: " + reply)
    _log(sd, "cgmas", reply)
    return {**state, "aa_fin_data_path": aa_fin_path, "aa_log_path": result["log_path"], "status": "done"}

def route_after_definer(state: LAMMPSState) -> str:
    if state.get("status") == "defining_next_component":
        return "next_component"
    return "need_confirm" if state.get("sim_params") is None else "skip_confirm"

def route_after_aa_params(state: LAMMPSState) -> str:
    return "confirmed" if state["status"] == "params_confirmed" else "pending"

def route_after_aa_review(state: LAMMPSState) -> str:
    if state["status"] == "approved":
        return "approved"
    elif state["revision_count"] >= MAX_REVISIONS:
        print("  Max revisions (" + str(MAX_REVISIONS) + ") reached. Using current file.")
        return "approved"
    return "revise"

def route_after_sim_params(state: LAMMPSState) -> str:
    return "confirmed" if state["status"] == "sim_confirmed" else "pending"

def route_after_aa_run(state: LAMMPSState) -> str:
    return "done" if state["status"] == "done" else "waiting"

def retry_revision(state: LAMMPSState) -> LAMMPSState:
    return {**state, "revision_count": state["revision_count"] + 1}


In [ ]:
# Bead mapping nodes

MAP_DESIGN_SYSTEM = """You are an expert in coarse-grained (CG) molecular simulation.

The user wants to define bead mapping from an all-atom polymer simulation.

Polymer info:
  Name        : {polymer_name}
  Atoms/monomer: {n_per_monomer}
  Monomers/chain: {n_monomers}
  Atom types  : {atom_type_info}

Output EXACTLY one valid JSON object (no markdown, no extra text):
{{
  "description": "<short description of the mapping>",
  "bead_types": [
    {{
      "bead_type_id":           <int, 1-based>,
      "name":                   "<label e.g. backbone, ring>",
      "monomer_type":           "<COPOLYMER ONLY: which monomer this bead maps, e.g. A or B; OMIT for homopolymer>",
      "local_ids_in_monomer":   [<1-based atom indices within ITS monomer>],
      "repeating_monomers":     <int, usually 1>
    }}
  ]
}}

Rules:
- local_ids_in_monomer must use 1-based indices (1 = first atom in monomer).
- All atoms should be assigned to exactly one bead type.
- If user says "1 monomer = 1 bead", assign all atom indices to one bead type.
- If user says "2 monomers = 1 bead", set repeating_monomers=2 and include all atom indices.
- For multi-bead-type (e.g. backbone + ring), define separate bead types with non-overlapping local_ids.
- Use the user description to determine the correct grouping.
{copoly_rule}"""


def map_design_node(state: LAMMPSState) -> LAMMPSState:
    sd = state["session_dir"]

    _is_retry = state.get("status") == "bead_retry"

    if not _is_retry:
        done_msg = "Simulation done! Files saved."
        if _log_once(sd, "cgmas", done_msg):  
            print("\n" + "="*60)
            print("CGMas: " + done_msg)
            print("="*60)

        ask_msg = "How would you like to map the beads? (e.g. 1 monomer = 1 bead, or backbone + ring as separate beads)"
        if _log_once(sd, "cgmas", ask_msg):   
            print("\nCGMas: " + ask_msg)

    user_reply = _resume(state, "map_design")
    _log(sd, "user", user_reply)

    poly_def   = state["poly_def"]
    sim_params = state["sim_params"] or {}
    n_monomers = sim_params.get("n_monomers", 10)

    from polymer import normalize_poly_def
    _pd = normalize_poly_def(poly_def, n_monomers)
    _mtypes = _pd.get("monomer_types", {})
    _is_copoly = (poly_def.get("polymer_type") == "copolymer") or (len(_mtypes) > 1)

    if _is_copoly:
        blocks = []
        for lbl, mdef in _mtypes.items():
            atoms_str = ", ".join(
                str(a["local_id"]) + "=" + a.get("comment", a["opls_id"])
                for a in mdef.get("monomer_atoms", [])
            )
            blocks.append(f"  monomer '{lbl}' ({mdef.get('name', lbl)}), "
                          f"{len(mdef.get('monomer_atoms', []))} atoms: {atoms_str}")
        atom_type_info = "COPOLYMER -- atoms listed per monomer type:\n" + "\n".join(blocks)
        n_per_mono = "varies by monomer type (see list)"
        copoly_rule = (
            "- THIS IS A COPOLYMER. Every bead_type MUST include a \"monomer_type\" field "
            "naming which monomer it maps (one of: "
            + ", ".join(repr(k) for k in _mtypes.keys()) + ").\n"
            "- local_ids_in_monomer are 1-based indices WITHIN that bead's own monomer type.\n"
            "- For \"1 monomer = 1 bead\", output exactly ONE bead_type per monomer type, "
            "each covering ALL atoms of that monomer; do NOT mix atoms from different monomers."
        )
    else:
        atom_type_info = ", ".join(
            str(a["local_id"]) + "=" + a.get("comment", a["opls_id"])
            for a in poly_def.get("monomer_atoms", [])
        )
        n_per_mono = len(poly_def.get("monomer_atoms", []))
        copoly_rule = ""

    system_prompt = MAP_DESIGN_SYSTEM.format(
        polymer_name   = poly_def.get("polymer_name", "unknown"),
        n_per_monomer  = n_per_mono,
        n_monomers     = n_monomers,
        atom_type_info = atom_type_info,
        copoly_rule    = copoly_rule,
    )
    resp = llm.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=user_reply),
    ])
    raw = resp.content.strip()
    clean = re.sub(r'^```(?:json)?\s*', '', raw)
    clean = re.sub(r'```\s*$', '', clean).strip()

    try:
        bead_def = json.loads(clean)
        if not (isinstance(bead_def, dict) and bead_def.get("bead_types")):
            raise ValueError("parsed mapping has no bead_types")
    except Exception:
        retry_msg = (
            "Sorry, I couldn't read that as a bead-mapping instruction. "
            "Please describe how to group the atoms into beads -- for example "
            "\"1 monomer = 1 bead\", \"2 monomers = 1 bead\", or "
            "\"backbone and ring as separate beads\"."
        )
        print("\nCGMas: " + retry_msg)
        _log(sd, "cgmas", retry_msg)
        return {**state, "bead_def": None, "status": "bead_retry"}

    return {**state, "bead_def": bead_def, "status": "bead_defined"}

def map_confirm_node(state: LAMMPSState) -> LAMMPSState:
    sd = state["session_dir"]
    bead_def = state["bead_def"]
    poly_def = state["poly_def"]

    lines = []
    for bt in bead_def["bead_types"]:
        reps = bt.get("repeating_monomers", 1)
        atom_names = []
        for lid in bt["local_ids_in_monomer"]:
            ma = next((a for a in poly_def.get("monomer_atoms", []) if a["local_id"] == lid), None)
            if ma:
                atom_names.append(ma.get("comment", ma.get("opls_id", str(lid))))
        reps_str = f" x{reps} monomers" if reps > 1 else ""
        lines.append(f"bead {bt['bead_type_id']} ({bt['name']}): {', '.join(atom_names)}{reps_str}")

    confirm_msg = bead_def.get("description", "") + "\n  " + "\n  ".join(lines) + "\nConfirm?"
    if _log_once(sd, "cgmas", confirm_msg):  
        print("\n" + "="*60)
        print("CGMas: " + confirm_msg)
        print("="*60)

    user_reply = _resume(state, "map_confirm", auto_reply="yes, the bead mapping is correct, approve")
    _log(sd, "user", user_reply)

    resp = llm.invoke([
        SystemMessage(content="Does the user approve? (yes/ok/go on/confirm) -> APPROVED, else -> CHANGE. Only APPROVED or CHANGE."),
        HumanMessage(content=user_reply),
    ])
    if "APPROVED" in resp.content.upper():
        reply = "Starting bead mapping..."
        print("\nCGMas: " + reply)
        _log(sd, "cgmas", reply)
        return {**state, "status": "bead_confirmed"}
    else:
        return {**state, "bead_def": None, "status": "bead_pending"}


def mapper_node(state: LAMMPSState) -> LAMMPSState:
    sd       = state["session_dir"]
    poly_def = state["poly_def"]
    bead_def = state["bead_def"]
    params   = state["sim_params"] or {}
    
    bead_map_def = {"bead_types": []}
    for bt in bead_def.get("bead_types", []):
        entry = {
            "bead_type_id":     bt["bead_type_id"],
            "name":             bt.get("name", "bead"),
            "description":      bt.get("description", bt.get("name", "bead")),
            "atom_local_ids":   bt.get("local_ids_in_monomer", bt.get("atom_local_ids", [])),
            "monomers_per_bead": bt.get("repeating_monomers", bt.get("monomers_per_bead", 1)),
        }
        if bt.get("monomer_type") is not None:
            entry["monomer_type"] = bt["monomer_type"]
        bead_map_def["bead_types"].append(entry)

    import glob
    fin_candidates = (
        glob.glob(os.path.join(sd, "AAEQfin*.data")) +
        glob.glob(os.path.join(sd, "*fin*.data"))
    )
    if not fin_candidates:
        fin_candidates = [state.get("final_datafile_path") or state.get("datafile_path")]
    aa_data_path = fin_candidates[0]

    poly_name    = re.sub(r"[^\w]+", "_", state.get("polymer_name", "polymer").lower())
    cg_data_path = os.path.join(sd, poly_name + "_bead.data")

    print("\n" + "="*60)
    print("[BeadMapper] Building CG data file...")
    print("  AA data  : " + str(aa_data_path))
    print("  CG data  : " + cg_data_path)
    print("="*60)

    build_cg_data(
        aa_data_path = aa_data_path,
        poly_def     = poly_def,
        bead_map_def = bead_map_def,
        output_path  = cg_data_path,
        n_chains     = params.get("n_chains", 20),
        n_monomers   = params.get("n_monomers", 10),
    )

    reply = "Bead mapping complete. CG data file: " + os.path.basename(cg_data_path)
    print("\nCGMas: " + reply)
    _log(sd, "cgmas", reply)

    return {**state, "cg_data_path": cg_data_path, "status": "bead_mapped"}


def route_after_map_confirm(state: LAMMPSState) -> str:
    if state["status"] == "bead_confirmed": return "confirmed"
    return "pending"


def route_after_map_design(state: LAMMPSState) -> str:
    if state["status"] == "bead_retry": return "retry"
    return "defined"


In [ ]:
# Potential calculation nodes

def pot_gate_node(state: LAMMPSState) -> LAMMPSState:
    sd = state["session_dir"]
    done_msg = "Bead mapping complete. CG data file: " + os.path.basename(state["cg_data_path"])
    ask_msg  = "Proceed with distribution function calculation for CG potential derivation?"
    if _log_once(sd,"cgmas",done_msg):
        print("\n"+"="*60)
        print("CGMas: "+done_msg)
        print("CGMas: "+ask_msg)
        print("="*60)
    _log_once(sd,"cgmas",ask_msg)
    user_reply = _resume(state, "pot_gate", auto_reply="yes, proceed with the distribution function calculation for the CG potential")
    _log(sd,"user",user_reply)
    resp = llm.invoke([SystemMessage(content="Does user want to proceed? (yes/ok->APPROVED, else SKIP). Only APPROVED or SKIP."),HumanMessage(content=user_reply)])
    if "APPROVED" in resp.content.upper():
        print("\nCGMas: Calculating CG potential parameters...")
        _log(sd,"cgmas","Calculating CG potential parameters...")
        return {**state, "status":"potential_confirmed"}
    print("\nCGMas: Skipping potential calculation.")
    return {**state, "status":"potential_skipped"}


CG_PAIR_CUTOFF = 10.0

def _aa_target_density(state):
    log_path = state.get("aa_log_path")
    if log_path and os.path.exists(log_path):
        try:
            from analyzer import parse_npt_log, calc_equilibrium_density
            return float(calc_equilibrium_density(parse_npt_log(log_path, label="aa")))
        except Exception as e:
            print(f"[Potential] AA density from log failed ({e}); "
                  "using the AA data file geometry instead.")
    return None   

def potential_node(state: LAMMPSState) -> LAMMPSState:
    sd=state["session_dir"]; poly_def=state["poly_def"]; params=state["sim_params"]
    print("\n"+"="*60+"\n[Potential] Calculating...\n"+"="*60)
    import glob
    trj = (glob.glob(os.path.join(sd,"aa_fin.lammpstrj")) or
           glob.glob(os.path.join(sd,"*fin*.lammpstrj")) or
           glob.glob(os.path.join(sd,"fin.lammpstrj")))
    if not trj:
        raise FileNotFoundError(
            "No equilibration trajectory found in " + sd +
            ". Expected: aa_fin.lammpstrj or *fin*.lammpstrj")
    if not trj: raise FileNotFoundError("No lammpstrj in "+sd)
    trj_path = trj[0]
    aa_fin = state.get("aa_fin_data_path")
    if not aa_fin:
        cands = glob.glob(os.path.join(sd,"*fin*.data"))
        aa_fin = cands[0] if cands else os.path.join(sd,"AAEQfin.data")
    _dirs  = state.get("directives") or {}
    _simsp = state.get("sim_script_params") or {}
    sim_T  = (_dirs.get("pot_temperature")
              or _dirs.get("aa_t_init")
              or _simsp.get("t_init")
              or DEFAULT_T_INIT)
    print(f"[Potential] Boltzmann inversion temperature: {float(sim_T):.1f} K")
    raw_bead = state["bead_def"]
    def _mk_bt(bt):
        e = {"bead_type_id":bt["bead_type_id"],"name":bt.get("name","bead"),
             "description":bt.get("description",bt.get("name","bead")),
             "atom_local_ids":bt.get("local_ids_in_monomer",bt.get("atom_local_ids",[])),
             "monomers_per_bead":bt.get("repeating_monomers",bt.get("monomers_per_bead",1))}
        if bt.get("monomer_type") is not None:
            e["monomer_type"] = bt["monomer_type"]
        if "is_backbone" in bt:
            e["is_backbone"] = bt["is_backbone"]
        return e
    bead_map_def = {"bead_types":[_mk_bt(bt) for bt in raw_bead.get("bead_types",[])]}
    cg_topology = state.get("cg_topology")
    if cg_topology:
        results = compute_cg_potentials_multi(
            trj_path=trj_path,aa_data_path=aa_fin,poly_def=poly_def,bead_map_def=bead_map_def,
            cg_topology={"bond_type_pairs":[tuple(p) for p in cg_topology["bond_type_pairs"]],
                "angle_type_triplets":[tuple(t) for t in cg_topology["angle_type_triplets"]],
                "lj_type_pairs":[tuple(p) for p in cg_topology["lj_type_pairs"]]},
            n_chains=params["n_chains"],n_monomers=params["n_monomers"],
            temperature=float(sim_T),frame_stride=1,output_dir=sd,
            rdf_exclude_depth=3, energy_matching=True,
            target_density_gcc=_aa_target_density(state),
            cg_pair_cutoff=CG_PAIR_CUTOFF)
    else:
        results = compute_cg_potentials(
            trj_path=trj_path,aa_data_path=aa_fin,poly_def=poly_def,bead_map_def=bead_map_def,
            n_chains=params["n_chains"],n_monomers=params["n_monomers"],
            temperature=float(sim_T),frame_stride=1,output_dir=sd)
    return {**state,"potential_results":_serialize_potential_results(results),"status":"potential_done"}


In [ ]:
# Potential reviewer + user confirm agents

MAX_POTENTIAL_REVISIONS = 2

POT_REVIEW_SYSTEM = 'You are an expert in CG molecular simulation potentials.\n\nGiven cg_topology (all expected types) and potential_results (computed values),\nperform TWO checks:\n\nCHECK 1 - MISSING TYPES:\n  Compare cg_topology lists vs results keys. Any type in topology but absent\n  from results keys -> add to add_*_types. This is the most common error.\n\nCHECK 2 - BAD VALUES:\n  - theta0 > 180 degrees -> impossible -> REMOVE that angle type\n  - r0 < 1.5 A for CG bonds -> unphysical (CG beads represent multiple atoms)\n  - k=0 AND (r0=0 or theta0=0) -> no data, type does not exist -> REMOVE\n  - epsilon=0 for LJ -> insufficient data, keep but note\n\nCRITICAL: BB-BB-BB angle (e.g.[1,1,1]) MUST always exist if chain has 3+ backbone beads.\nIf missing from results, add it.\n\nOutput one valid JSON (no markdown):\n{"status": "APPROVED" or "REVISE",\n "issues": ["..."],\n "remove_bond_types": [[ti,tj],...],\n "remove_angle_types": [[ti,tj,tk],...],\n "remove_lj_types": [[ti,tj],...],\n "add_bond_types": [[ti,tj],...],\n "add_angle_types": [[ti,tj,tk],...],\n "add_lj_types": [[ti,tj],...],\n "action": "REMOVE_TYPES" or "ADD_TYPES" or "REMOVE_ADD" or "RECOMPUTE" or "NONE"}'

POT_CONFIRM_SYSTEM = 'Parse the user reply about CG potential results.\nCurrent topology: {topo}\nIf user approves (ok/yes/confirm/looks good) -> APPROVED.\nIf user requests changes, return JSON:\n{"action": "MODIFY",\n "remove_bond_types": [[ti,tj],...],\n "remove_angle_types": [[ti,tj,tk],...],\n "remove_lj_types": [[ti,tj],...],\n "add_bond_types": [[ti,tj],...],\n "add_angle_types": [[ti,tj,tk],...],\n "add_lj_types": [[ti,tj],...]}\nRespond ONLY APPROVED or JSON.'


def _apply_topo_changes(topo, review):
    new_topo = json.loads(json.dumps(topo))
    def _tt(lst): return [tuple(x) for x in lst]
    rb=_tt(review.get('remove_bond_types',[])); ra=_tt(review.get('remove_angle_types',[]))
    rl=_tt(review.get('remove_lj_types',[]));   ab=review.get('add_bond_types',[])
    aa=review.get('add_angle_types',[]);          al=review.get('add_lj_types',[])
    if rb: new_topo['bond_type_pairs']=[p for p in topo['bond_type_pairs'] if tuple(p) not in rb]
    if ra: new_topo['angle_type_triplets']=[t for t in topo['angle_type_triplets'] if tuple(t) not in ra]
    if rl: new_topo['lj_type_pairs']=[p for p in topo['lj_type_pairs'] if tuple(p) not in rl]
    if ab:
        ex=[tuple(p) for p in new_topo['bond_type_pairs']]
        for p in [sorted(x) for x in ab]:
            if tuple(p) not in ex: new_topo['bond_type_pairs'].append(p)
    if aa:
        ex=[tuple(t) for t in new_topo['angle_type_triplets']]
        for t in [list(x) for x in aa]:
            if tuple(t) not in ex: new_topo['angle_type_triplets'].append(t)
    if al:
        ex=[tuple(p) for p in new_topo['lj_type_pairs']]
        for p in [sorted(x) for x in al]:
            if tuple(p) not in ex: new_topo['lj_type_pairs'].append(p)
    return new_topo, rb, ra, rl, ab, aa, al

def _is_multi_pot(d: dict) -> bool:
    """True if d uses tuple/list/tuple-string keys (multi bead-type potential result)."""
    if not d: return False
    k = next(iter(d))
    if isinstance(k, (tuple, list)): return True
    if isinstance(k, str) and (k.startswith('(') or k.startswith('[')): return True
    return False

def _sanitize_for_state(obj):
    """
    Recursively convert numpy scalars / arrays to native Python types so that
    LangGraph's MemorySaver (msgpack backend) can serialize the state without
    raising "Type is not msgpack serializable: numpy.float64".
    Also converts numpy scalar dict keys to native int/float.
    """
    import numpy as np
    def _key(k):
        if isinstance(k, np.integer):  return int(k)
        if isinstance(k, np.floating): return float(k)
        return k
    if isinstance(obj, dict):
        return {_key(k): _sanitize_for_state(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        converted = [_sanitize_for_state(v) for v in obj]
        return converted if isinstance(obj, list) else tuple(converted)
    if isinstance(obj, np.integer):  return int(obj)
    if isinstance(obj, np.floating): return float(obj)
    if isinstance(obj, np.ndarray):  return obj.tolist()
    if isinstance(obj, np.bool_):    return bool(obj)
    return obj

def _serialize_potential_results(results: dict) -> dict:
    """
    Convert tuple keys -> "[ti, tj]" strings so LangGraph MemorySaver
    (JSON-backed) can checkpoint the state without TypeError.
    All consumers handle string keys via _to_tuple / _is_multi_pot.
    """
    if not results:
        return results
    bond = results.get('bond', {})
    if not _is_multi_pot(bond):
        return results 

    def _k(t): return str(list(t)) if isinstance(t, tuple) else str(t)

    serialized = {
        'bond':             {_k(k): v for k, v in results['bond'].items()},
        'angle':            {_k(k): v for k, v in results['angle'].items()},
        'lj':               {_k(k): v for k, v in results['lj'].items()},
        'bond_type_order':  [list(k) if isinstance(k, tuple) else k
                             for k in (results.get('bond_type_order')
                                       or results['bond'].keys())],
        'angle_type_order': [list(k) if isinstance(k, tuple) else k
                             for k in (results.get('angle_type_order')
                                       or results['angle'].keys())],
        'lj_type_order':    [list(k) if isinstance(k, tuple) else k
                             for k in (results.get('lj_type_order')
                                       or results['lj'].keys())],
        **{k: v for k, v in results.items()
           if k not in ('bond','angle','lj','bond_type_order','angle_type_order','lj_type_order')},
    }
    return _sanitize_for_state(serialized)

def _key_to_list(k):
    """Convert any key form (tuple, list, '[1,1]' string) to display list."""
    if isinstance(k, (tuple, list)): return list(k)
    if isinstance(k, str):
        import ast
        try: return list(ast.literal_eval(k))
        except: return k
    return k


def _format_potential_msg(results):
    bond_res=results['bond']; angle_res=results['angle']; lj_res=results['lj']
    is_multi=(_is_multi_pot(bond_res) or _is_multi_pot(angle_res) or _is_multi_pot(lj_res))
    if is_multi:
        lines=[]
        for bt,p in bond_res.items():  lines.append(f"bond type {_key_to_list(bt)}: r0={p['r0']:.3f} \u00c5, k={p['k']:.3f} kcal/mol/\u00c5\u00b2")
        for at,p in angle_res.items(): lines.append(f"angle type {_key_to_list(at)}: \u03b80={p['theta0']:.3f}\u00b0, k={p['k']:.3f} kcal/mol/rad\u00b2")
        for lj,p in lj_res.items():    lines.append(f"LJ type {_key_to_list(lj)}: \u03c3={p['sigma']:.3f} \u00c5, \u03b5={p['epsilon']:.3f} kcal/mol")
        return '\n'.join(lines)
    b,a,lj=bond_res,angle_res,lj_res
    return f"bond: r0={b['r0']:.3f} \u00c5, k={b['k']:.3f}  |  angle: \u03b80={a['theta0']:.3f}\u00b0, k={a['k']:.3f}  |  LJ: \u03c3={lj['sigma']:.3f} \u00c5, \u03b5={lj['epsilon']:.3f}"


def pot_review_node(state: LAMMPSState) -> LAMMPSState:
    sd=state['session_dir']; results=state['potential_results']; topo=state.get('cg_topology',{})
    print('\n'+'='*60+'\n[Potential Reviewer] Auto-checking results...\n'+'='*60)
    bond_res=results['bond']; angle_res=results['angle']; lj_res=results['lj']
    is_multi=(_is_multi_pot(bond_res) or _is_multi_pot(angle_res) or _is_multi_pot(lj_res))
    def _ser(d): return {str(_key_to_list(k)):v for k,v in d.items()}
    results_str=json.dumps({'bond':_ser(bond_res),'angle':_ser(angle_res),'lj':_ser(lj_res)} if is_multi else results,indent=2)
    resp=llm.invoke([SystemMessage(content=POT_REVIEW_SYSTEM),
        HumanMessage(content='cg_topology:\n'+json.dumps(topo,indent=2)+'\n\npotential_results:\n'+results_str)])
    raw=resp.content.strip()
    clean=re.sub(r'^```(?:json)?\s*','',raw); clean=re.sub(r'```\s*$','',clean).strip()
    try: review=json.loads(clean)
    except Exception: review={'status':'APPROVED','issues':[],'action':'NONE'}
    print('[Potential Reviewer] Status:', review.get('status'))
    for issue in review.get('issues',[]): print('  Issue:', issue)
    rev_count=state.get('potential_revision_count',0)
    if review.get('status')=='APPROVED' or rev_count>=MAX_POTENTIAL_REVISIONS:
        if rev_count>=MAX_POTENTIAL_REVISIONS: print('[Reviewer] Max revisions reached.')
        return {**state,'status':'potential_reviewed'}
    new_topo,rb,ra,rl,ab,aa,al=_apply_topo_changes(topo,review)
    locked = state.get('user_locked_topology') or {'bonds':[],'angles':[],'lj':[]}
    rb = [t for t in rb if list(t) not in locked['bonds']]
    ra = [t for t in ra if list(t) not in locked['angles']]
    rl = [t for t in rl if list(t) not in locked['lj']]
    _tb = {tuple(sorted(p)) for p in topo.get('bond_type_pairs', [])}
    _ta = set()
    for _t in topo.get('angle_type_triplets', []):
        _ta.add(tuple(_t)); _ta.add(tuple(reversed(_t)))
    _tl = {tuple(sorted(p)) for p in topo.get('lj_type_pairs', [])}
    _kept_b = [t for t in rb if tuple(sorted(t)) in _tb]
    _kept_a = [t for t in ra if tuple(t) in _ta]
    _kept_l = [t for t in rl if tuple(sorted(t)) in _tl]
    if _kept_b or _kept_a or _kept_l:
        print('[Reviewer] Ignoring removal of data-backed types still in topology: '
              f'bonds={_kept_b} angles={_kept_a} LJ={_kept_l}')
    rb = [t for t in rb if tuple(sorted(t)) not in _tb]
    ra = [t for t in ra if tuple(t) not in _ta]
    rl = [t for t in rl if tuple(sorted(t)) not in _tl]
    filtered_review = {**review, 'remove_bond_types': rb, 'remove_angle_types': ra, 'remove_lj_types': rl}
    new_topo, *_ = _apply_topo_changes(topo, filtered_review)
    all_empty = not any([rb, ra, rl, ab, aa, al])
    if all_empty:
        print('[Reviewer] No actionable topology changes. Skipping recompute.')
        _log(sd,'cgmas','[Reviewer] No actionable topology changes.')
        return {**state,'potential_revision_count':rev_count+1,'status':'potential_reviewed'}
    fix_msg=f'[Reviewer] Removed bonds:{rb} angles:{ra} LJ:{rl}\n           Added bonds:{ab} angles:{aa} LJ:{al}\nRecomputing...'
    print(fix_msg); _log(sd,'cgmas',fix_msg)
    return {**state,'cg_topology':new_topo,'potential_revision_count':rev_count+1,'status':'potential_recompute'}


def pot_confirm_node(state: LAMMPSState) -> LAMMPSState:
    sd=state['session_dir']; results=state['potential_results']; topo=state.get('cg_topology',{})
    msg=_format_potential_msg(results)
    confirm_msg=msg+'\nConfirm? (or request changes e.g. \'recalculate the LJ\', \'add angle type [1,1,1]\')'
    if _log_once(sd,'cgmas',confirm_msg):        # guard: skip on LangGraph resume re-run
        print('\n'+'='*60)
        print('CGMas: '+confirm_msg)
        print('='*60)
    user_reply=_resume(state, 'pot_confirm', auto_reply="approve, the potentials look good")
    _log(sd,'user',user_reply)
    prompt = (
        'User replied to CG potential results.\n'
        'Current topology: ' + json.dumps(topo) + '\n'
        'User reply: ' + user_reply + '\n\n'
        'There are THREE possible responses:\n'
        '1) User APPROVES (ok/yes/looks good/confirm) -> respond exactly: APPROVED\n'
        '2) User wants to RECOMPUTE specific existing types without topology change\n'
           '   (e.g. "recalculate angle (1,1,1)", "redo lj 2,2", "recompute bond")\n'
           '   -> respond exactly: RECOMPUTE\n'
        '3) User wants to ADD or REMOVE types -> respond with JSON:\n'
        '{"action":"MODIFY","add_angle_types":[[1,1,1]],'
        '"remove_bond_types":[],"remove_angle_types":[],"remove_lj_types":[],'
        '"add_bond_types":[],"add_lj_types":[]}\n'
        'Respond ONLY with APPROVED, RECOMPUTE, or valid JSON. No other text.'
    )
    parse_resp=llm.invoke([HumanMessage(content=prompt)])

    raw = parse_resp.content.strip()
    clean = re.sub(r'^```(?:json)?\s*', '', raw)
    clean = re.sub(r'```\s*$', '', clean).strip()

    if clean == 'APPROVED':
        redo_resp = llm.invoke([HumanMessage(content=
            'Does the user EXPLICITLY want to redo/restart bead mapping from scratch? '
            'Only YES if they use words like: redo bead mapping, restart mapping, '
            'bead mapping is wrong, start over mapping. '
            'A plain yes/ok/confirm does NOT count. '
            'Respond only YES or NO. User said: ' + user_reply)])
        if 'YES' in redo_resp.content.upper():
            msg2 = 'Restarting from bead mapping...'
            print('\nCGMas: ' + msg2); _log(sd,'cgmas',msg2)
            return {**state, 'bead_def': None, 'cg_topology': None,
                    'potential_results': None, 'user_locked_topology': None,
                    'potential_revision_count': 0, 'status': 'redo_bead_mapping'}
        reply = 'Potential parameters confirmed. Proceeding to CG simulation...'
        print('\nCGMas: ' + reply); _log(sd, 'cgmas', reply)
        return {**state, 'status': 'potential_reviewed'}

    if clean == 'RECOMPUTE':
        msg2 = 'Understood. Recomputing with the current topology...'
        print('\nCGMas: ' + msg2); _log(sd, 'cgmas', msg2)
        return {**state, 'potential_revision_count': 0, 'status': 'potential_recompute'}

    try:
        changes = json.loads(clean)
        new_topo, rb, ra, rl, ab, aa, al = _apply_topo_changes(topo, changes)
        locked = json.loads(json.dumps(state.get('user_locked_topology') or {'bonds': [], 'angles': [], 'lj': []}))
        for p in changes.get('add_bond_types',  []): locked['bonds'].append(list(p))
        for t in changes.get('add_angle_types', []): locked['angles'].append(list(t))
        for p in changes.get('add_lj_types',    []): locked['lj'].append(list(p))
        fix_msg = ('[User] Removed bonds:' + str(rb) + ' angles:' + str(ra) + ' LJ:' + str(rl)
                   + '\n       Added bonds:' + str(ab) + ' angles:' + str(aa) + ' LJ:' + str(al)
                   + '\nRecomputing...')
        print(fix_msg); _log(sd, 'cgmas', fix_msg)
        return {**state, 'cg_topology': new_topo, 'user_locked_topology': locked,
                'potential_revision_count': 0, 'status': 'potential_recompute'}
    except Exception as e:
        print('[potential_user_confirm] Parse error, treating as APPROVED:', e)
        return {**state, 'status': 'potential_reviewed'}


In [ ]:
# CG topology agent node

TOPOLOGY_SYSTEM = 'Given poly_def and bead_def, determine ALL CG bond/angle/LJ types.\nBond types: directly bonded bead pairs. BB-BB always. BB-pendant if exists.\nAngle types: ALL triplets from actual connectivity:\n  - BB-BB-BB always (consecutive backbone beads)\n  - BB-BB-P and P-BB-BB if pendants attach to backbone\nLJ types: ALL unique unordered pairs (every bead type combination).\nOutput one valid JSON (no markdown):\n{"bond_type_pairs":[[1,1],...],"angle_type_triplets":[[1,1,1],[1,1,2],[2,1,1],...],"lj_type_pairs":[[1,1],[1,2],[2,2],...]}\nSorted order for pairs (smaller id first). Keep angle triplet direction as given.'


def _cg_types_from_data(path):
    """Read a mapped CG LAMMPS data file and return the unique bond bead-type
    pairs and angle bead-type triplets actually present (ground truth for which
    coeffs the CG constructor will need)."""
    bonds, angles, atom_type = [], [], {}
    if not path or not os.path.exists(path):
        return bonds, angles
    section = None
    with open(path) as f:
        for line in f:
            t = line.split('#')[0].strip()
            if not t:
                continue
            if t in ('Atoms', 'Bonds', 'Angles', 'Masses', 'Velocities',
                     'Bond Coeffs', 'Angle Coeffs', 'Pair Coeffs'):
                section = t
                continue
            parts = t.split()
            try:
                if section == 'Atoms':
                    atom_type[int(parts[0])] = int(parts[2])
                elif section == 'Bonds':
                    a, b = int(parts[2]), int(parts[3])
                    bonds.append((atom_type.get(a), atom_type.get(b)))
                elif section == 'Angles':
                    a, b, c = int(parts[2]), int(parts[3]), int(parts[4])
                    angles.append((atom_type.get(a), atom_type.get(b), atom_type.get(c)))
            except (ValueError, IndexError):
                continue
    uniq_b = sorted({tuple(sorted(p)) for p in bonds if None not in p})
    def _norm_ang(t):
        e = tuple(sorted((t[0], t[2])))
        return (e[0], t[1], e[1])
    uniq_a = sorted({_norm_ang(t) for t in angles if None not in t})
    return [list(p) for p in uniq_b], [list(t) for t in uniq_a]

def topology_node(state: LAMMPSState) -> LAMMPSState:
    sd=state['session_dir']
    print('\n'+'='*60+'\n[CG Topology] Inferring bond/angle/LJ types...\n'+'='*60)
    resp=llm.invoke([SystemMessage(content=TOPOLOGY_SYSTEM),
        HumanMessage(content='poly_def:\n'+json.dumps(state['poly_def'],indent=2)
                     +'\n\nbead_def:\n'+json.dumps(state['bead_def'],indent=2))])
    raw=resp.content.strip()
    clean=re.sub(r'^```(?:json)?\s*','',raw); clean=re.sub(r'```\s*$','',clean).strip()
    cg_topology=json.loads(clean)
    cg_topology['bond_type_pairs']     =[sorted(p) for p in cg_topology.get('bond_type_pairs',[])]
    cg_topology['angle_type_triplets'] =[list(t)   for t in cg_topology.get('angle_type_triplets',[])]
    cg_topology['lj_type_pairs']       =[sorted(p) for p in cg_topology.get('lj_type_pairs',[])]
    bead_ids=sorted(set(bt['bead_type_id'] for bt in state['bead_def'].get('bead_types',[])))
    all_lj=[sorted([i,j]) for i in bead_ids for j in bead_ids if i<=j]
    ex_lj=[sorted(p) for p in cg_topology['lj_type_pairs']]
    for p in all_lj:
        if p not in ex_lj: cg_topology['lj_type_pairs'].append(p)
    bb=bead_ids[0]
    _db, _da = [], []
    try:
        _db, _da = _cg_types_from_data(state.get('cg_data_path'))
    except Exception:
        _db, _da = [], []
    if _db:
        cg_topology['bond_type_pairs'] = [sorted(p) for p in _db]
    elif not cg_topology['bond_type_pairs']:
        cg_topology['bond_type_pairs'] = [[bb, bb]]
    if _da:
        _seen, _ang = set(), []
        for _t in _da:
            _key = min(tuple(_t), tuple(_t[::-1]))
            if _key not in _seen:
                _seen.add(_key); _ang.append(list(_t))
        cg_topology['angle_type_triplets'] = _ang
    elif not cg_topology['angle_type_triplets']:
        cg_topology['angle_type_triplets'] = [[bb, bb, bb]]
    msg=('CG topology determined:\n'
         '  Bond types  : '+str(cg_topology['bond_type_pairs'])+'\n'
         '  Angle types : '+str(cg_topology['angle_type_triplets'])+'\n'
         '  LJ types    : '+str(cg_topology['lj_type_pairs']))
    print('\nCGMas: '+msg); _log(sd,'cgmas',msg)
    return {**state,'cg_topology':cg_topology,'status':'cg_topology_ready'}


In [ ]:
# CG Simulation nodes

AVOGADRO_NB = 6.02214076e23

def _cg_density_from_box(box_L, n_chains, n_monomers, bead_def, bead_masses):
    max_mpb = max(bt.get("monomers_per_bead", 1) for bt in bead_def["bead_types"])
    n_groups = n_monomers // max_mpb
    total_mass = 0.0
    for _ in range(n_groups):
        for bt in bead_def["bead_types"]:
            total_mass += bead_masses[bt["bead_type_id"]]
    total_mass *= n_chains
    V_cm3 = (box_L * 1e-8) ** 3
    return total_mass / (AVOGADRO_NB * V_cm3)


def _aa_last_step_density(log_path):
    """AA LAMMPS log에서 가장 마지막(가장 최근) step의 Density 값을 반환합니다.
    파싱 실패 시 None."""
    if not log_path:
        return None
    try:
        lines = open(log_path, errors="ignore").read().splitlines()
    except Exception:
        return None
    _hdr = re.compile(r"^\s*Step\s+", re.IGNORECASE)
    dens_idx = None
    last_dens = None
    for ln in lines:
        if _hdr.match(ln):
            cols = ln.split()
            dens_idx = next((k for k, c in enumerate(cols)
                             if c.lower() == "density"), None)
            continue
        if dens_idx is None:
            continue
        toks = ln.split()
        if len(toks) <= dens_idx:
            continue
        try:
            float(toks[0])  # step 컬럼이 숫자여야 데이터 행
            val = float(toks[dens_idx].replace("D", "E").replace("d", "e"))
        except ValueError:
            continue
        last_dens = val
    return last_dens

def cg_gate_node(state: LAMMPSState) -> LAMMPSState:
    sd = state["session_dir"]
    msg = "Ready to start CG simulation. Type \'ok. let\'s start coarse-grained simulation.\' to proceed."
    if _log_once(sd, "cgmas", msg):              # guard: skip on LangGraph resume re-run
        print("\nCGMas: " + msg)

    user_reply = _resume(state, "cg_gate", auto_reply="ok, let's start the coarse-grained simulation")
    _log(sd, "user", user_reply)

    resp = llm.invoke([
        SystemMessage(content=(
            "Does the user want to start coarse-grained simulation? "
            "(ok/yes/start/cg simulation/coarse-grained) -> YES, else NO. Only YES or NO."
        )),
        HumanMessage(content=user_reply),
    ])
    if "YES" in resp.content.upper():
        return {**state, "status": "cg_init"}
    else:
        msg2 = "CG simulation not started. Let me know when ready."
        print("\nCGMas: " + msg2)
        _log(sd, "cgmas", msg2)
        return {**state, "status": "cg_waiting_trigger"}

def cg_init_node(state: LAMMPSState) -> LAMMPSState:
    sd = state["session_dir"]
    msg = "Got it. Building CG topology..."
    print("\n" + "="*60)
    print("CGMas: " + msg)
    print("="*60)
    _log(sd, "cgmas", msg)
    return {**state, "status": "cg_building"}

def cg_params_node(state: LAMMPSState) -> LAMMPSState:
    sd = state["session_dir"]
    bead_def = state["bead_def"]

    aa_params  = state.get("sim_params") or {}
    prev       = state.get("cg_sim_params") or {}
    n_chains   = prev.get("n_chains",   aa_params.get("n_chains",   20))
    n_monomers = prev.get("n_monomers", aa_params.get("n_monomers", 10))
    density    = prev.get("density")
    if density is None:
        _aa_d = _aa_last_step_density(state.get("aa_log_path"))
        density = round(_aa_d, 1) if _aa_d is not None else aa_params.get("density", 0.9)

    opls = load_opls(OPLS_DIR)
    from cg_constructor import _bead_type_masses, _calculate_cg_box_size
    bead_masses = _bead_type_masses(state["poly_def"], bead_def, opls["atom"])
    box_L = round(_calculate_cg_box_size(n_chains, n_monomers, bead_def, bead_masses, density), 3)

    params = {"n_chains": n_chains, "n_monomers": n_monomers, "density": density, "box_size": box_L}

    msg = (
        "chain length: " + str(n_monomers) + ", "
        "number of chains: " + str(n_chains) + ", "
        "density: " + str(density) + " g/cm\u00b3, "
        "box size: " + str(box_L) + " \u00c5. Confirm?"
    )
    if _log_once(sd, "cgmas", msg):  
        print("\n" + "="*60)
        print("CGMas: " + msg)
        print("="*60)

    user_reply = _resume(state, "cg_params", auto_reply=_auto_param_reply(state, "cg_params"))
    _log(sd, "user", user_reply)

    parse_resp = llm.invoke([
        SystemMessage(content=(
            "Parse the user reply about CG simulation parameters."
            " Current params: " + json.dumps(params) + "."
            " If user approves (ok/yes/go/confirm) -> APPROVED."
            " Otherwise ONLY JSON with changed fields (n_chains, n_monomers, density, box_size)."
            " Respond ONLY APPROVED or JSON. No other text."
        )),
        HumanMessage(content=user_reply),
    ])
    parsed = parse_resp.content.strip()

    if parsed == "APPROVED":
        reply = "Confirmed. Proceeding to data file construction..."
        print("\nCGMas: " + reply)
        _log(sd, "cgmas", reply)
        return {**state, "cg_sim_params": params, "status": "cg_params_confirmed"}

    try:
        changes = json.loads(re.sub(r'^```(?:json)?\s*|```\s*$', '', parsed).strip())
        updated = {**params, **changes}
        if "box_size" in changes and "density" not in changes:
            updated["density"] = round(
                _cg_density_from_box(updated["box_size"], updated["n_chains"],
                                     updated["n_monomers"], bead_def, bead_masses), 4)
        elif "box_size" not in changes:
            from cg_constructor import _calculate_cg_box_size
            updated["box_size"] = round(
                _calculate_cg_box_size(updated["n_chains"], updated["n_monomers"],
                                       bead_def, bead_masses, updated["density"]), 3)
    except Exception:
        updated = params

    return {**state, "cg_sim_params": updated, "status": ("cg_params_confirmed" if state.get("auto") else "cg_params_pending")}

def cg_constructor_node(state: LAMMPSState) -> LAMMPSState:
    sd       = state["session_dir"]
    params   = state["cg_sim_params"]
    bead_def = state["bead_def"]
    poly_def = state["poly_def"]
    pot      = state["potential_results"]

    print("\n" + "="*60)
    print("[CG Constructor] Generating CG data file...")
    print("="*60)

    opls = load_opls(OPLS_DIR)
    poly_name = re.sub(r"[^\w]+", "_", state.get("polymer_name", "polymer").lower())
    data_path = os.path.join(sd, poly_name + "_cg_ini.data")

    build_result = build_cg_initial_data(
        poly_def          = poly_def,
        bead_def          = bead_def,
        potential_results = pot,
        opls_atoms        = opls["atom"],
        n_chains          = params["n_chains"],
        n_monomers        = params["n_monomers"],
        density           = params["density"],
        output_path       = data_path,
        cg_topology       = state.get("cg_topology"),
    )

    reply = "CG data file created. (" + os.path.basename(data_path) + ")"
    print("\nCGMas: " + reply)
    _log(sd, "cgmas", reply)
    return {**state, "cg_initial_data_path": data_path,
            "cg_data_build_meta": _sanitize_for_state(build_result),
            "status": "cg_data_ready"}

def cg_sim_params_node(state: LAMMPSState) -> LAMMPSState:
    sd = state["session_dir"]

    prev     = state.get("cg_sim_script_params") or {}
    t_init   = prev.get("t_init",   DEFAULT_T_INIT_CG)
    t_max    = prev.get("t_max",    DEFAULT_T_MAX_CG)
    n_anneal = prev.get("n_anneal", DEFAULT_N_ANNEAL_CG)
    equil_ps = prev.get("equil_ps", DEFAULT_EQUIL_PS_CG)
    params   = {"t_init": t_init, "t_max": t_max, "n_anneal": n_anneal, "equil_ps": equil_ps}
    params   = _apply_temp_directives(state, "cg_sim_params", params)
    t_init, t_max, n_anneal = params["t_init"], params["t_max"], params["n_anneal"]

    cycle_word = "cycles" if n_anneal > 1 else "cycle"
    msg = (
        "Operating temp.: " + str(t_init) + " K, "
        "Up temp.: " + str(t_max) + " K, "
        "annealing: " + str(n_anneal) + " " + cycle_word + ", "
        "Total equilibrium time: " + str(equil_ps) + " ps. Confirm?"
    )
    if _log_once(sd, "cgmas", msg):    
        print("\n" + "="*60)
        print("CGMas: " + msg)
        print("="*60)

    user_reply = _resume(state, "cg_sim_params", auto_reply=_auto_param_reply(state, "cg_sim_params"))
    _log(sd, "user", user_reply)

    parse_resp = llm.invoke([
        SystemMessage(content=(
            "Parse the user reply about CG LAMMPS simulation parameters."
            " Current params: " + json.dumps(params) + "."
            " If user approves (ok/yes/go/confirm) -> APPROVED."
            " Otherwise ONLY JSON with changed fields."
            " e.g. max temp 450K -> t_max=450, 2ns eq -> equil_ps=2000."
            " Respond ONLY APPROVED or JSON. No other text."
        )),
        HumanMessage(content=user_reply),
    ])
    parsed = parse_resp.content.strip()

    if parsed == "APPROVED":
        reply = "Confirmed. Generating CG simulation script..."
        print("\nCGMas: " + reply)
        _log(sd, "cgmas", reply)
        return {**state, "cg_sim_script_params": params, "status": "cg_sim_confirmed"}

    try:
        changes = json.loads(re.sub(r'^```(?:json)?\s*|```\s*$', '', parsed).strip())
        updated = {**params, **changes}
    except Exception:
        updated = params
    updated = _apply_temp_directives(state, "cg_sim_params", updated)

    return {**state, "cg_sim_script_params": updated, "status": ("cg_sim_confirmed" if state.get("auto") else "cg_sim_pending")}

def cg_sim_build_node(state: LAMMPSState) -> LAMMPSState:
    sd       = state["session_dir"]
    params   = state["cg_sim_script_params"]
    pot      = state["potential_results"]
    bead_def = state["bead_def"]
    poly_def = state["poly_def"]

    print("\n" + "="*60)
    print("[CG SimConstructor] Generating CG simulation script...")
    print("="*60)

    data_filename  = os.path.basename(state["cg_initial_data_path"])
    poly_name = re.sub(r"[^\w]+", "_", state.get("polymer_name", "polymer").lower())
    in_file_path   = os.path.join(sd, "in." + poly_name + "_CGEQ")

    _meta = state.get("cg_data_build_meta") or {}
    build_cg_in_file(
        template_path     = CG_TEMPLATE_IN,
        data_filename     = data_filename,
        potential_results = pot,
        bead_def          = bead_def,
        output_path       = in_file_path,
        t_init            = params["t_init"],
        t_max             = params["t_max"],
        n_anneal          = params["n_anneal"],
        equil_ps          = params["equil_ps"],
        used_bond_types   = _meta.get("used_bond_types"),
        used_angle_types  = _meta.get("used_angle_types"),
        bond_type_remap   = _meta.get("bond_type_remap"),
        angle_type_remap  = _meta.get("angle_type_remap"),
    )
    print(f"[CG SimConstructor] CG temperatures written to {os.path.basename(in_file_path)}: "
          f"Tini = {params['t_init']} K, Tfin = {params['t_max']} K")

    reply   = "Script construction complete. (" + os.path.basename(in_file_path) + ")"
    run_msg = "Ready to simulate. Type \'OK. please simulate.\' to start LAMMPS."
    print("\nCGMas: " + reply)
    print("CGMas: " + run_msg)
    _log(sd, "cgmas", reply)
    _log(sd, "cgmas", run_msg)

    return {**state, "cg_in_file_path": in_file_path, "status": "cg_script_ready"}

def cg_run_node(state: LAMMPSState) -> LAMMPSState:
    sd = state["session_dir"]

    user_reply = _resume(state, "cg_run", auto_reply="yes, please run the LAMMPS simulation now")
    _log(sd, "user", user_reply)

    resp = llm.invoke([
        SystemMessage(content=(
            "Does the user want to start LAMMPS CG simulation? "
            "If yes (simulate/run/go/start/ok/please simulate) respond: YES. "
            "Otherwise respond: NO. Only YES or NO."
        )),
        HumanMessage(content=user_reply),
    ])
    wants_run = "YES" in resp.content.upper()

    if not wants_run:
        msg = "CG simulation not started. Let me know when you\'re ready."
        print("\nCGMas: " + msg)
        _log(sd, "cgmas", msg)
        return {**state, "status": "cg_waiting_run"}

    print("\n" + "="*60)
    print("[CG LAMMPS] Starting CG simulation...")
    print("="*60)

    result = run_lammps(
        in_file    = state["cg_in_file_path"],
        datafile   = state["cg_initial_data_path"],
        output_dir = sd,
        n_cores    = 12,
        label      = "cg",
    )

    if result["returncode"] == 0:
        reply = (
            "CG simulation done! Files saved in " + os.path.basename(sd) + "/  "
            "-- Log: " + os.path.basename(result["log_path"]) + ", "
            "Screen: " + os.path.basename(result["screen_path"])
        )
    else:
        reply = "CG LAMMPS error (code " + str(result["returncode"]) + ")."

    print("\nCGMas: " + reply)
    _log(sd, "cgmas", reply)
    return {**state, "cg_log_path": result["log_path"], "status": "cg_done"}

def route_after_cg_params(state: LAMMPSState) -> str:
    return "confirmed" if state["status"] == "cg_params_confirmed" else "pending"

def route_after_cg_sim_params(state: LAMMPSState) -> str:
    return "confirmed" if state["status"] == "cg_sim_confirmed" else "pending"

def route_after_cg_run(state: LAMMPSState) -> str:
    return "done" if state["status"] == "cg_done" else "waiting"

def _deserialize_pot_keys(pot: dict) -> dict:
    """Convert state string-keys back to tuple-keys.
    Every other entry is carried through untouched. That matters because
    potential_results also holds bond_type_order / angle_type_order /
    lj_type_order, and build_cg_in_file drives its coeff loops from those lists.
    Dropping them here used to be harmless (the consumer fell back to the dict
    keys when the entry was ABSENT), but any caller that serialises the result
    back into state reintroduces the entries as EMPTY lists -- and an empty list
    silences the fallback, so the CG input file ends up with no bond_coeff,
    angle_coeff or pair_coeff at all and LAMMPS aborts with
    "All pair coeffs are not set".
    """
    import ast
    def _tk(k):
        if isinstance(k, tuple): return k
        if isinstance(k, list): return tuple(k)
        if isinstance(k, str):
            try: return tuple(ast.literal_eval(k))
            except: return k
        return k
    bond = pot.get("bond", {})
    if not bond: return pot
    first = next(iter(bond))
    if isinstance(first, str) and (first.startswith("[") or first.startswith("(")):
        out = dict(pot)          
        for sect in ("bond", "angle", "lj"):
            if isinstance(pot.get(sect), dict):
                out[sect] = {_tk(k): v for k, v in pot[sect].items()}
        for sect in ("bond_type_order", "angle_type_order", "lj_type_order"):
            if isinstance(pot.get(sect), list):
                out[sect] = [_tk(k) for k in pot[sect]]
        return out
    return pot


def tg_gate_node(state: LAMMPSState) -> LAMMPSState:
    """Wait for user to request annealing/thermal simulation.

    Robustness: an ambiguous or unrecognised reply (e.g. a mistyped command
    like 'tg' / 'calculate tg' when the user meant 'start thermal annealing')
    must NOT end the workflow. Only an explicit request to finish/skip ends it;
    anything else re-prompts so the user can type their query again.
    """
    sd = state["session_dir"]

    msg = ("Ready for Tg annealing (AA first, then CG). Type 'start thermal "
           "annealing' to begin, or 'skip' to finish the workflow.")
    if _log_once(sd, "cgmas", msg):    
        print("\nCGMas: " + msg)

    user_reply = _resume(state, "tg_gate", auto_reply="yes, run the annealing thermal Tg glass transition simulation now")
    _log(sd, "user", user_reply)

    resp = llm.invoke([
        SystemMessage(content=(
            "Classify the user's reply at the glass-transition (Tg) annealing "
            "step. Respond with EXACTLY one word: START, SKIP, or UNCLEAR.\n"
            "- START: they want to start / run / proceed with annealing, thermal, "
            "or Tg simulation, OR they mention annealing / thermal / Tg / glass "
            "transition / 'calculate tg' / 'looking good' in any form.\n"
            "- SKIP: ONLY if they EXPLICITLY want to finish, skip, stop, quit, or "
            "end the workflow (e.g. 'skip', 'done', 'finish', 'no thanks', "
            "'that's all', 'end').\n"
            "- UNCLEAR: anything ambiguous or that matches neither."
        )),
        HumanMessage(content=user_reply),
    ])
    intent = resp.content.strip().upper()

    if "START" in intent:
        return {**state, "status": "tg_confirmed"}
    if "SKIP" in intent:
        msg = "Skipping Tg annealing. Workflow complete."
        print("\nCGMas: " + msg); _log(sd, "cgmas", msg)
        return {**state, "status": "tg_skipped"}

    msg = ("Sorry, I didn't catch that. Type 'start thermal annealing' to begin "
           "Tg annealing, or 'skip' to finish the workflow.")
    print("\nCGMas: " + msg); _log(sd, "cgmas", msg)
    return {**state, "status": "tg_waiting"}

def _tg_ramp_temps(state):
    """Reference-Tg-derived (Tini, Tfin) for the annealing ramp.

    Resolved once and cached in state so the AA and CG runs anneal over the
    SAME temperature window -- otherwise the two Tg values would not be
    comparable. Returns (t_ini, t_fin, tg_temps_dict).
    """
    cached = state.get("tg_temps")
    if cached:
        return cached["t_ini"], cached["t_fin"], cached

    polymer_name = state.get("polymer_name", "polymer")
    ref_tg = get_reference_tg(polymer_name, llm)
    t_ini, t_fin = calc_tg_temps(ref_tg["tg_K"])
    tg_temps = {"t_ini": int(t_ini), "t_fin": int(t_fin),
                "ref_tg_K": float(ref_tg["tg_K"])}
    return t_ini, t_fin, tg_temps


def _cg_tg_datafile(state):
    """CG structure for Tg annealing.

    Prefer the CG-equilibrated file, then the CG initial data. NEVER fall back
    to a bare *fin*.data glob: that would match the all-atom AAEQfin.data
    (which carries dihedrals) and, read under the CG "atom_style angle",
    triggers "No dihedrals allowed with this atom style".
    """
    import glob as _glob
    sd = state["session_dir"]
    cg_fin = (_glob.glob(os.path.join(sd, "CGEQfin.data")) or
              _glob.glob(os.path.join(sd, "*CGEQ*fin*.data")) or
              _glob.glob(os.path.join(sd, "*_cg_fin.data")))
    return cg_fin[0] if cg_fin else (state.get("cg_initial_data_path") or "")


def _aa_tg_datafile(state):
    """AA structure for Tg annealing -- the equilibrated AAEQfin.data."""
    import glob as _glob
    sd = state["session_dir"]
    aa_fin = state.get("aa_fin_data_path")
    if aa_fin and os.path.exists(aa_fin):
        return aa_fin
    cands = (_glob.glob(os.path.join(sd, "AAEQfin.data")) or
             _glob.glob(os.path.join(sd, "*AAEQ*fin*.data")) or
             _glob.glob(os.path.join(sd, "*_aa_fin.data")))
    return cands[0] if cands else ""

def aa_tg_build_node(state: LAMMPSState) -> LAMMPSState:
    """Generate in.AATg from the AA template, running on AAEQfin.data.

    Same patching as the AA equilibration step (build_in_file): Tini/Tfin
    variables, read_data, and the element list on the dump_modify line. AA
    force-field coefficients are NOT touched -- they live in the data file.
    """
    sd = state["session_dir"]

    msg = "Starting AA thermal annealing. Generating simulation script..."
    print("\n" + "=" * 60)
    print("CGMas: " + msg)
    print("=" * 60)
    _log(sd, "cgmas", msg)

    t_ini, t_fin, tg_temps = _tg_ramp_temps(state)

    poly_name   = re.sub(r"[^\w]+", "_", state.get("polymer_name", "polymer").lower())
    in_filename = "in." + poly_name + "_AATg"
    in_path     = os.path.join(sd, in_filename)

    if not os.path.exists(AA_TG_TEMPLATE_IN):
        err = (f"AA Tg template not found: {os.path.basename(AA_TG_TEMPLATE_IN)} "
               f"(looked in {SCRIPT_DIR}). Skipping AA annealing.")
        print("\nCGMas: " + err); _log(sd, "cgmas", err)
        return {**state, "tg_temps": tg_temps, "aa_tg_in_path": None,
                "status": "aa_tg_skipped"}

    datafile = _aa_tg_datafile(state)
    if not datafile:
        err = ("AA Tg annealing needs AAEQfin.data from the AA equilibration run, "
               "but none was found. Skipping AA annealing.")
        print("\nCGMas: " + err); _log(sd, "cgmas", err)
        return {**state, "tg_temps": tg_temps, "aa_tg_in_path": None,
                "status": "aa_tg_skipped"}

    elements_src = state.get("final_datafile_path") or state.get("datafile_path")

    print(f"[AA Tg] Patching template : {os.path.basename(AA_TG_TEMPLATE_IN)}")
    print(f"[AA Tg] Structure         : {os.path.basename(datafile)}")

    try:
        build_aa_tg_in_file(
            template_path = AA_TG_TEMPLATE_IN,
            output_path   = in_path,
            datafile      = datafile,
            t_ini         = t_ini,
            t_fin         = t_fin,
            elements_from = elements_src,
        )
    except Exception as e:
        err = f"AA Tg script generation failed ({e}). Skipping AA annealing."
        print("\nCGMas: " + err); _log(sd, "cgmas", err)
        return {**state, "tg_temps": tg_temps, "aa_tg_in_path": None,
                "status": "aa_tg_skipped"}

    msg2 = f"Script file construction complete. ({in_filename})"
    print("\nCGMas: " + msg2); _log(sd, "cgmas", msg2)
    return {**state, "tg_temps": tg_temps, "aa_tg_in_path": in_path,
            "status": "aa_tg_built"}

def aa_tg_run_node(state: LAMMPSState) -> LAMMPSState:
    """Confirm, then run the AA Tg annealing. Log -> lammps_aa_tg.log."""
    sd = state["session_dir"]

    msg = "Ready to simulate AA annealing. Type 'OK. please simulate.' to start LAMMPS."
    if _log_once(sd, "cgmas", msg): 
        print("\nCGMas: " + msg)

    user_reply = _resume(state, "aa_tg_run", auto_reply="yes, please simulate now")
    _log(sd, "user", user_reply)

    resp = llm.invoke([
        SystemMessage(content=(
            "Does the user want to start LAMMPS simulation? "
            "YES for: simulate/run/go/start/ok/please simulate. "
            "Only YES or NO."
        )),
        HumanMessage(content=user_reply),
    ])
    if "YES" not in resp.content.upper():
        msg2 = "Simulation not started. Let me know when you're ready."
        print("\nCGMas: " + msg2); _log(sd, "cgmas", msg2)
        return {**state, "status": "aa_tg_waiting_run"}

    print("\n" + "=" * 60)
    print("[LAMMPS] Starting AA Tg annealing simulation...")
    print("=" * 60)

    result = run_lammps(
        in_file    = state["aa_tg_in_path"],
        datafile   = _aa_tg_datafile(state),
        output_dir = sd,
        n_cores    = 12,
        label      = "aa_tg",    
    )

    if result["returncode"] == 0:
        reply = "AA annealing done! Now starting CG thermal annealing."
    else:
        reply = ("AA annealing LAMMPS error (code " + str(result["returncode"]) +
                 "). Continuing to CG thermal annealing.")

    print("\nCGMas: " + reply); _log(sd, "cgmas", reply)
    return {**state, "aa_tg_log_path": result["log_path"], "status": "aa_tg_run_done"}

def cg_tg_build_node(state: LAMMPSState) -> LAMMPSState:
    """Generate in.CGTg using the CG potential parameters and the same ramp."""
    sd = state["session_dir"]

    msg = "Starting CG thermal annealing. Generating simulation script..."
    print("\n" + "=" * 60)
    print("CGMas: " + msg)
    print("=" * 60)
    _log(sd, "cgmas", msg)

    t_ini, t_fin, tg_temps = _tg_ramp_temps(state)

    poly_name   = re.sub(r"[^\w]+", "_", state.get("polymer_name", "polymer").lower())
    in_filename = "in." + poly_name + "_CGTg"
    in_path     = os.path.join(sd, in_filename)

    potential_results = _deserialize_pot_keys(state.get("potential_results") or {})
    datafile = _cg_tg_datafile(state)

    build_tg_in_file(
        template_path     = TG_TEMPLATE_IN,
        output_path       = in_path,
        potential_results = potential_results,
        t_ini             = t_ini,
        t_fin             = t_fin,
        datafile          = datafile,
    )

    msg2 = f"Script file construction complete. ({in_filename})"
    print("\nCGMas: " + msg2); _log(sd, "cgmas", msg2)
    return {**state, "tg_temps": tg_temps, "tg_in_path": in_path, "status": "tg_built"}

def cg_tg_run_node(state: LAMMPSState) -> LAMMPSState:
    """Confirm, then run the CG Tg annealing. Log -> lammps_cg_tg.log."""
    sd = state["session_dir"]

    msg = "Ready to simulate CG annealing. Type 'OK. please simulate.' to start LAMMPS."
    if _log_once(sd, "cgmas", msg):    
        print("\nCGMas: " + msg)

    user_reply = _resume(state, "tg_run", auto_reply="yes, please simulate now")
    _log(sd, "user", user_reply)

    resp = llm.invoke([
        SystemMessage(content=(
            "Does the user want to start LAMMPS simulation? "
            "YES for: simulate/run/go/start/ok/please simulate. "
            "Only YES or NO."
        )),
        HumanMessage(content=user_reply),
    ])
    if "YES" not in resp.content.upper():
        msg2 = "Simulation not started. Let me know when you're ready."
        print("\nCGMas: " + msg2); _log(sd, "cgmas", msg2)
        return {**state, "status": "tg_waiting_run"}

    print("\n" + "=" * 60)
    print("[LAMMPS] Starting CG Tg annealing simulation...")
    print("=" * 60)

    result = run_lammps(
        in_file    = state["tg_in_path"],
        datafile   = _cg_tg_datafile(state),
        output_dir = sd,
        n_cores    = 12,
        label      = "cg_tg",    
    )

    if result["returncode"] == 0:
        reply = "Simulation done!"
    else:
        reply = "LAMMPS error (code " + str(result["returncode"]) + ")."

    print("\nCGMas: " + reply); _log(sd, "cgmas", reply)
    return {**state, "tg_log_path": result["log_path"], "status": "tg_run_done"}

tg_build_node = cg_tg_build_node
tg_run_node   = cg_tg_run_node


def tg_calc_node(state: LAMMPSState) -> LAMMPSState:
    """Compute Tg from BOTH cooling ramps: the AA log and the CG log.

    Each Tg comes from the '# ----- Temp decrease' (cooling) segment of its own
    log, so the two numbers are independent estimates over the same temperature
    window. Figures: aa_tg_fit.png and cg_tg_fit.png.
    """
    sd = state["session_dir"]

    user_reply = _resume(state, "tg_calc", auto_reply="yes, calculate the glass transition temperature Tg now")
    _log(sd, "user", user_reply)

    resp = llm.invoke([
        SystemMessage(content=(
            "Does the user want to calculate the glass transition temperature (Tg)? "
            "Keywords: calculate, Tg, glass transition, temperature. "
            "Only YES or NO."
        )),
        HumanMessage(content=user_reply),
    ])
    if "YES" not in resp.content.upper():
        return {**state, "status": "tg_calc_waiting"}

    results = {}
    for key, lbl, title in [("aa_tg_log_path", "aa", "AA"),
                            ("tg_log_path",    "cg", "CG")]:
        log_path = state.get(key)
        if not (log_path and os.path.exists(log_path)):
            print(f"[tg_calc] {title}: annealing log not found -- skipped")
            continue
        try:
            print(f"\n[tg_calc] {title} cooling ramp  ({os.path.basename(log_path)})")
            results[lbl] = float(calc_tg_from_annealing(
                log_path   = log_path,
                output_dir = sd,
                label      = lbl,
                show       = True,
            ))
        except Exception as e:
            print(f"[tg_calc] {title}: Tg calculation failed: {e}")

    if not results:
        reply = "Tg calculation failed: no usable annealing log for AA or CG."
        print("\nCGMas: " + reply); _log(sd, "cgmas", reply)
        return {**state, "status": "tg_calc_done"}

    parts = []
    if "aa" in results: parts.append(f"AA Tg: {results['aa']:.1f} K")
    if "cg" in results: parts.append(f"CG Tg: {results['cg']:.1f} K")
    reply = "Tg calculation done. " + ", ".join(parts)
    print("\nCGMas: " + reply); _log(sd, "cgmas", reply)

    cg_tg = results.get("cg")
    return {**state,
            "aa_sim_tg": results.get("aa"),
            "cg_sim_tg": cg_tg,
            "sim_tg":    cg_tg if cg_tg is not None else results.get("aa"),
            "status":    "tg_calc_done"}

def tg_validate_node(state: LAMMPSState) -> LAMMPSState:
    """Final summary: AA vs CG self-consistency. No experimental reference.

    The question this answers is "does the CG model reproduce the all-atom
    system it was parametrised from?", so AA is the baseline and CG is scored
    against it. Experimental density/Tg lookups carry their own uncertainty and
    would blur that judgement, so they are not used here.

        density : |CG - AA| / AA x 100   -- PASS within +-5 %
        Tg      : |CG - AA|              -- PASS within +-15 K
    """
    sd = state["session_dir"]
    polymer_name = state.get("polymer_name", "Unknown")

    user_reply = _resume(state, "tg_validate", auto_reply="yes, give me the validation summary of the results")
    _log(sd, "user", user_reply)

    resp = llm.invoke([
        SystemMessage(content=(
            "Does the user want a validation summary? "
            "Keywords: summary, validation, validate, results. "
            "Only YES or NO."
        )),
        HumanMessage(content=user_reply),
    ])
    if "YES" not in resp.content.upper():
        return {**state, "status": "tg_validate_waiting"}

    aa_density = cg_density = None
    for log_key, lbl in [("aa_log_path", "aa"), ("cg_log_path", "cg")]:
        log_path = state.get(log_key)
        if log_path and os.path.exists(log_path):
            try:
                df   = parse_npt_log(log_path, label=lbl)
                dens = calc_equilibrium_density(df)
                if lbl == "aa":
                    aa_density = dens
                else:
                    cg_density = dens
            except Exception as _e:
                print(f"[Summary] Warning: could not parse {lbl} log: {_e}")

    aa_tg = state.get("aa_sim_tg")
    cg_tg = state.get("cg_sim_tg", state.get("sim_tg"))
    d_cmp  = compare_aa_cg_density(aa_density, cg_density)
    tg_cmp = compare_aa_cg_tg(aa_tg, cg_tg)

    sep   = "=" * 60
    lines = [sep, "  FINAL VALIDATION SUMMARY", sep,
             f"  Polymer      : {polymer_name}", ""]

    lines.append("  [Density]")
    if aa_density is not None:
        lines.append(f"    AA result  : {aa_density:.3f} g/cm\u00b3")
    else:
        lines.append("    AA result  : n/a")
    if cg_density is not None:
        tail = ""
        if d_cmp["error_pct"] is not None:
            tail = f"  ({d_cmp['error_pct']:.1f}% err)  {d_cmp['verdict']}"
        elif d_cmp["verdict"]:
            tail = f"  {d_cmp['verdict']}"
        lines.append(f"    CG result  : {cg_density:.3f} g/cm\u00b3{tail}")
    else:
        lines.append("    CG result  : n/a")

    lines += ["", "  [Tg]"]
    if aa_tg is not None:
        lines.append(f"    AA anneal  : {float(aa_tg):.1f} K")
    else:
        lines.append("    AA anneal  : n/a")
    if cg_tg is not None:
        tail = ""
        if tg_cmp["error_K"] is not None:
            tail = f"  ({tg_cmp['error_K']:.1f} K err)  {tg_cmp['verdict']}"
        lines.append(f"    CG anneal  : {float(cg_tg):.1f} K{tail}")
    else:
        lines.append("    CG anneal  : n/a")
    lines.append(sep)

    summary = "\n".join(lines)
    print("\n" + summary)
    _log(sd, "cgmas", summary)
    return {**state,
            "validation_summary": {"density": d_cmp, "tg": tg_cmp},
            "status": "tg_validate_done"}


def route_after_tg_gate(state: LAMMPSState) -> str:
    s = state["status"]
    if s == "tg_confirmed": return "confirmed"
    if s == "tg_skipped":   return "skip"
    return "waiting"

def route_after_aa_tg_build(state: LAMMPSState) -> str:
    return "skip" if state["status"] == "aa_tg_skipped" else "built"

def route_after_aa_tg_run(state: LAMMPSState) -> str:
    return "done" if state["status"] == "aa_tg_run_done" else "waiting"

def route_after_tg_run(state: LAMMPSState) -> str:
    return "done" if state["status"] == "tg_run_done" else "waiting"

def route_after_tg_calc(state: LAMMPSState) -> str:
    return "done" if state["status"] == "tg_calc_done" else "waiting"

def route_after_tg_validate(state: LAMMPSState) -> str:
    return "done" if state["status"] == "tg_validate_done" else "waiting"

In [ ]:
def plots_node(state: LAMMPSState) -> LAMMPSState:
    """Save density / temperature / potential-energy vs time figures for AA and
    CG (individually and as an AA-vs-CG overlay) after the validation summary."""
    from analyzer import parse_npt_log, plot_npt_properties

    sd = state["session_dir"]
    print("\n" + "=" * 60)
    print("[Plots] Saving NPT property graphs (density / temperature / PE vs time)...")
    print("=" * 60)

    dfs = {}
    for log_key, lbl in [("aa_log_path", "aa"), ("cg_log_path", "cg")]:
        log_path = state.get(log_key)
        if not (log_path and os.path.exists(log_path)):
            print(f"[plots] {lbl}: log not found ({log_path}) -- skipped")
            continue
        try:
            dfs[lbl] = parse_npt_log(log_path, label=lbl)
        except Exception as e:
            print(f"[plots] {lbl}: could not parse NPT log: {e}")

    saved = []
    for lbl, df in dfs.items():
        try:
            saved += plot_npt_properties(df, output_dir=sd, prefix=f"{lbl}_", show=False)
        except Exception as e:
            print(f"[plots] {lbl}: plotting failed: {e}")
    if len(dfs) >= 2:
        try:
            saved += plot_npt_properties(*dfs.values(), output_dir=sd,
                                         prefix="aa_vs_cg_", show=False)
        except Exception as e:
            print(f"[plots] AA-vs-CG overlay failed: {e}")

    try:
        from potential import plot_distributions
        saved += plot_distributions(output_dir=sd, show=False)
    except Exception as e:
        print(f"[plots] distribution plots skipped: {e}")

    tg_pngs = sorted(f for f in os.listdir(sd)
                     if f.endswith("tg_fit.png")) if os.path.isdir(sd) else []

    if saved:
        print("[plots] saved:")
        for p in saved:
            print("   - " + os.path.basename(p))
    for f in tg_pngs:
        print("   - " + f + "  (Tg curve, from tg_calc)")
    _log(sd, "cgmas",
         f"Plots saved: {len(saved)} NPT figures"
         + (f" + {', '.join(tg_pngs)}" if tg_pngs else "")
         + f" in {os.path.basename(sd)}/")
    return {**state, "status": "plots_done"}

In [ ]:
memory=MemorySaver()
workflow=StateGraph(LAMMPSState)

workflow.add_node("greet",              safe_node(greet_node))
workflow.add_node("copoly_spec",       safe_node(copolymer_spec_node))
workflow.add_node("definer",          safe_node(definer_agent))
workflow.add_node("aa_params",      safe_node(aa_params_node))
workflow.add_node("aa_constructor",        safe_node(aa_constructor_node))
workflow.add_node("aa_review",           safe_node(aa_review_agent))
workflow.add_node("retry",          safe_node(retry_revision))
workflow.add_node("sim_params",  safe_node(sim_params_node))
workflow.add_node("sim_build",    safe_node(sim_build_node))
workflow.add_node("aa_run",        safe_node(aa_run_node))
workflow.add_node("map_design",        safe_node(map_design_node))
workflow.add_node("map_confirm",       safe_node(map_confirm_node))
workflow.add_node("mapper",           safe_node(mapper_node))
workflow.add_node("pot_gate",  safe_node(pot_gate_node))
workflow.add_node("potential",     safe_node(potential_node))
workflow.add_node("pot_review", safe_node(pot_review_node))
workflow.add_node("pot_confirm", safe_node(pot_confirm_node))
workflow.add_node("topology",  safe_node(topology_node))
workflow.add_node("cg_gate",           safe_node(cg_gate_node))
workflow.add_node("cg_init",             safe_node(cg_init_node))
workflow.add_node("cg_params",     safe_node(cg_params_node))
workflow.add_node("cg_constructor",  safe_node(cg_constructor_node))
workflow.add_node("cg_sim_params", safe_node(cg_sim_params_node))
workflow.add_node("cg_sim_build",   safe_node(cg_sim_build_node))
workflow.add_node("cg_run",       safe_node(cg_run_node))

workflow.set_entry_point("greet")
workflow.add_edge("greet","copoly_spec")
workflow.add_edge("copoly_spec","definer")
workflow.add_conditional_edges("definer",route_after_definer,{"next_component":"definer","need_confirm":"aa_params","skip_confirm":"aa_constructor"})
workflow.add_conditional_edges("aa_params",route_after_aa_params,{"confirmed":"aa_constructor","pending":"aa_params"})
workflow.add_edge("aa_constructor","aa_review")
workflow.add_conditional_edges("aa_review",route_after_aa_review,{"approved":"sim_params","revise":"retry"})
workflow.add_edge("retry","definer")
workflow.add_conditional_edges("sim_params",route_after_sim_params,{"confirmed":"sim_build","pending":"sim_params"})
workflow.add_edge("sim_build","aa_run")
workflow.add_conditional_edges("aa_run",route_after_aa_run,{"done":"map_design","waiting":"aa_run"})
workflow.add_conditional_edges("map_design",route_after_map_design,{"defined":"map_confirm","retry":"map_design"})
workflow.add_conditional_edges("map_confirm",route_after_map_confirm,{"confirmed":"mapper","pending":"map_design"})
workflow.add_edge("mapper","topology")
workflow.add_edge("topology","pot_gate")
workflow.add_conditional_edges("pot_gate",
    lambda s:"confirmed" if s["status"]=="potential_confirmed" else "skip",
    {"confirmed":"potential","skip":"cg_gate"})
workflow.add_edge("potential","pot_review")
workflow.add_conditional_edges("pot_review",
    lambda s:"recompute" if s["status"]=="potential_recompute" else "done",
    {"done":"pot_confirm","recompute":"potential"})
workflow.add_conditional_edges("pot_confirm",
    lambda s:"redo" if s["status"]=="redo_bead_mapping" else ("recompute" if s["status"]=="potential_recompute" else "confirmed"),
    {"confirmed":"cg_gate","recompute":"potential","redo":"map_design"})
workflow.add_conditional_edges("cg_gate",
    lambda s:"start" if s["status"]=="cg_init" else "waiting",
    {"start":"cg_init","waiting":"cg_gate"})
workflow.add_edge("cg_init","cg_params")
workflow.add_conditional_edges("cg_params",route_after_cg_params,{"confirmed":"cg_constructor","pending":"cg_params"})
workflow.add_conditional_edges("cg_sim_params",route_after_cg_sim_params,{"confirmed":"cg_sim_build","pending":"cg_sim_params"})
workflow.add_edge("cg_sim_build","cg_run")
workflow.add_conditional_edges("cg_run",route_after_cg_run,{"done":"tg_gate","waiting":"cg_run"})

workflow.add_node("tg_gate",     safe_node(tg_gate_node))
workflow.add_node("aa_tg_build", safe_node(aa_tg_build_node))
workflow.add_node("aa_tg_run",   safe_node(aa_tg_run_node))
workflow.add_node("tg_build",    safe_node(cg_tg_build_node))
workflow.add_node("tg_run",      safe_node(cg_tg_run_node))
workflow.add_node("tg_calc",     safe_node(tg_calc_node))
workflow.add_node("tg_validate", safe_node(tg_validate_node))
workflow.add_node("plots", safe_node(plots_node))

workflow.add_conditional_edges("tg_gate", route_after_tg_gate,
    {"confirmed": "aa_tg_build", "skip": END, "waiting": "tg_gate"})
workflow.add_conditional_edges("aa_tg_build", route_after_aa_tg_build,
    {"built": "aa_tg_run", "skip": "tg_build"})
workflow.add_conditional_edges("aa_tg_run", route_after_aa_tg_run,
    {"done": "tg_build", "waiting": "aa_tg_run"})
workflow.add_edge("tg_build", "tg_run")
workflow.add_conditional_edges("tg_run", route_after_tg_run,
    {"done": "tg_calc", "waiting": "tg_run"})
workflow.add_conditional_edges("tg_calc", route_after_tg_calc,
    {"done": "tg_validate", "waiting": "tg_calc"})
workflow.add_conditional_edges("tg_validate", route_after_tg_validate,
    {"done": "plots", "waiting": "tg_validate"})
workflow.add_edge("plots", END)

app=workflow.compile(checkpointer=memory)


In [ ]:
# Execution function  --  interactive loop

def run_cgmas(initial_message:str='Hello, CGMas',thread_id:str=None)->str:
    if thread_id is None:
        import time
        thread_id = f'cgmas_{int(time.time())}'
    print('\nSession directory: '+session_dir)
    with tee_stdout(session_dir):
        return _run_cgmas_body(session_dir, thread_id, initial_message)


def _run_cgmas_body(session_dir, thread_id, initial_message):
    config={'configurable':{'thread_id':thread_id}}
    initial_state=LAMMPSState(
        session_dir=session_dir, initial_input=initial_message,
        polymer_name=None, sim_params=None, poly_def=None,
        datafile_path=None, review_feedback=None, final_datafile_path=None,
        sim_script_params=None, in_file_path=None, aa_fin_data_path=None,
        bead_map_def=None, bead_def=None, cg_data_path=None,
        potential_results=None, cg_sim_params=None, cg_initial_data_path=None,
        cg_in_file_path=None, cg_sim_script_params=None, cg_topology=None,
        user_locked_topology=None,
        revision_count=0, potential_revision_count=0,
        cg_data_build_meta=None,
        aa_log_path=None, cg_log_path=None,
        tg_temps=None, aa_tg_in_path=None, aa_tg_log_path=None,
        tg_in_path=None, tg_log_path=None,
        aa_sim_tg=None, cg_sim_tg=None, sim_tg=None,
        validation_summary=None,
        calibration=None,
        status='greeting',
        poly_defs=None, copolymer_spec=None, is_copolymer=False,
        _copoly_components=None, _copoly_current_idx=0,
    )
    try:
        for _ in app.stream(initial_state, config=config, stream_mode="updates"):
            pass
        while True:
            snapshot = app.get_state(config)
            if not snapshot.next:
                break
            user_input = input('You: ').strip()
            for _ in app.stream(Command(resume=user_input), config=config, stream_mode="updates"):
                pass
    except ValueError as e:
        print(f"\n{'='*60}\n[CGMas error] {e}\n{'='*60}")
        print("The session was interrupted.")
        return session_dir
    print('\n'+'='*60+'\nCGMas: Session complete.')
    print('       Directory: '+session_dir)
    print('       Chat log : '+os.path.join(session_dir,'chat_log.json')+'\n'+'='*60)
    return session_dir

def run_cgmas_auto(query: str, thread_id: str = None) -> str:
    """Run the ENTIRE pipeline (AA -> CG -> Tg -> validation summary -> plots)
    from a single user query, with no further human input. Anything not specified
    in the query (chain length/count, density, box size, temperatures, annealing
    cycles, ...) falls back to module defaults.
    """
    import time
    session_dir = make_session_dir()
    if thread_id is None:
        thread_id = f"cgmas_auto_{int(time.time())}"
    print("\nSession directory: " + session_dir)
    with tee_stdout(session_dir):
        return _run_cgmas_auto_body(session_dir, thread_id, query)


def _run_cgmas_auto_body(session_dir, thread_id, query):
    print("[AUTO] Running full pipeline from a single query -- no further input needed.\n")

    directives = parse_directives(query)
    _set = {k: v for k, v in directives.items() if v is not None}
    print("[AUTO] Parsed directives: " + (str(_set) if _set else "none -> all defaults"))

    tracker = TokenCostTracker()
    config = {"configurable": {"thread_id": thread_id}, "callbacks": [tracker],
              "recursion_limit": 200}
    initial_state = LAMMPSState(
        session_dir=session_dir, initial_input=query,
        polymer_name=None, sim_params=None, poly_def=None,
        datafile_path=None, review_feedback=None, final_datafile_path=None,
        sim_script_params=None, in_file_path=None, aa_fin_data_path=None,
        bead_map_def=None, bead_def=None, cg_data_path=None,
        potential_results=None, cg_sim_params=None, cg_initial_data_path=None,
        cg_in_file_path=None, cg_sim_script_params=None, cg_topology=None,
        user_locked_topology=None,
        revision_count=0, potential_revision_count=0,
        cg_data_build_meta=None,
        aa_log_path=None, cg_log_path=None,
        tg_temps=None, aa_tg_in_path=None, aa_tg_log_path=None,
        tg_in_path=None, tg_log_path=None,
        aa_sim_tg=None, cg_sim_tg=None, sim_tg=None,
        validation_summary=None,
        calibration=None,
        status="greeting",
        poly_defs=None, copolymer_spec=None, is_copolymer=False,
        _copoly_components=None, _copoly_current_idx=0,
        auto=True, initial_query=query, directives=directives,
    )
    try:
        for _ in app.stream(initial_state, config=config, stream_mode="updates"):
            pass
        guard = 0
        while guard < 50:
            snap = app.get_state(config)
            if not snap.next:
                break
            for _ in app.stream(Command(resume=query), config=config, stream_mode="updates"):
                pass
            guard += 1
    except ValueError as e:
        print(f"\n{'='*60}\n[CGMas AUTO error] {e}\n{'='*60}")
        _emit_run_summary(tracker, session_dir, ok=False)
        return session_dir

    _emit_run_summary(tracker, session_dir, ok=True)
    return session_dir


def _emit_run_summary(tracker, session_dir, ok=True):
    model_name = getattr(llm, "model_name", None) or getattr(llm, "model", "unknown")
    print("\n" + "=" * 60)
    if ok:
        print("CGMas: Autonomous session complete.")
    else:
        print("CGMas: Autonomous session ended early (see error above).")
    print("       Directory: " + session_dir)
    print("       Chat log : " + os.path.join(session_dir, "chat_log.json"))
    print("-" * 60)
    print(emit_token_report(tracker, session_dir, model_name))
    print("=" * 60)
